In [1]:
import scipy.io as sio
import numpy as np
from matplotlib.colors import ListedColormap
from scipy.stats import pearsonr
import seaborn as sn
import mne
import matplotlib.pyplot as plt
import copy
from scipy import stats
import pandas as pd
from statsmodels.stats.multitest import fdrcorrection
import copy
import matplotlib.pyplot as plt
import fooof
import specparam
from scipy.stats import permutation_test
from fooof import FOOOF
import numpy as np

%matplotlib qt
%gui qt

C:\Users\acadena\AppData\Local\Temp\ipykernel_24400\3990028331.py:14: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [ ]:
"""
Group-level EEG analysis: Rest vs. Mindfulness
==============================================

Extends the original topoplotsAverage_group with:
  1. Effect sizes (Cohen's d_z for paired data) per band/electrode
  2. Cluster-based permutation test (spatial) via MNE
  3. ROI analysis (frontal / central / parietal / occipital / temporal)
  4. Individual-subject variability plots
  5. Publication-ready plotting:
       - cleaner layout, perceptually uniform colormap (jet for diff, jet for power)
       - one summary figure per band
       - bar plots per band with individual-subject lines
       - consistent typography and spacing
  6. Automatic generation of a methods/figures log (figures_methods.txt)
     that documents how each figure and statistical result was produced,
     including the exact function names and libraries used.

Author: improved version
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import mne
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
from mne.stats import permutation_cluster_1samp_test
from scipy.stats import false_discovery_control, permutation_test, ttest_rel, wilcoxon

# -----------------------------------------------------------------------------
# Global plotting style
# -----------------------------------------------------------------------------
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "legend.frameon": False,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

FREQUENCY_BANDS: Dict[str, Tuple[float, float]] = {
    "Delta":     (0.5,  4),
    "Theta":     (4,    8),
    "Alpha":     (8,   12),
    "Beta":      (12,  18),
    "Fast Beta": (18,  30),
    "Gamma":     (30,  50),
}

# Standard 10-20 ROI definitions. Channels not in `info` are silently dropped.
ROI_DEFINITIONS: Dict[str, List[str]] = {
    "Frontal":   ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "AF3", "AF4"],
    "Central":   ["FC5", "FC1", "FC2", "FC6", "C3", "Cz", "C4", "CP5", "CP1", "CP2", "CP6"],
    "Parietal":  ["P7", "P3", "Pz", "P4", "P8", "PO3", "PO4"],
    "Occipital": ["O1", "Oz", "O2"],
    "Temporal":  ["T7", "T8", "FT7", "FT8", "TP7", "TP8"],
}


# =============================================================================
# Methods logger
# =============================================================================
class MethodsLogger:
    """
    Accumulates a human-readable log describing how each figure and
    statistical result was generated. Writes out a single .txt file.
    """

    def __init__(self):
        self.header_lines: List[str] = []
        self.entries: List[str] = []

    def set_header(self, **params):
        self.header_lines = [
            "=" * 78,
            "EEG group analysis — Figure generation & methods log",
            "=" * 78,
            f"Generated:        {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"MNE-Python:       {mne.__version__}",
            f"NumPy:            {np.__version__}",
            "",
            "Analysis parameters",
            "-" * 78,
        ]
        for k, v in params.items():
            self.header_lines.append(f"  {k:20s} = {v}")
        self.header_lines.append("")

    def add(self, title: str, lines: List[str]):
        block = [
            "=" * 78,
            title,
            "=" * 78,
        ]
        block.extend(lines)
        block.append("")
        self.entries.append("\n".join(block))

    def write(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(self.header_lines))
            f.write("\n")
            f.write("\n".join(self.entries))
        print(f"  Methods log saved to: {os.path.abspath(path)}")


# =============================================================================
# Data loading & PSD computation
# =============================================================================
@dataclass
class GroupPSD:
    """Container for group-level PSD data.

    Two representations are stored per subject:

    1) `psd_R`, `psd_MF` (n_subj, n_ch, n_freqs):
       Subject-level mean PSD spectra. If load_group_psd was called with
       relative=True, each subject's spectrum is the across-epoch mean of
       the per-epoch normalized PSDs (per-epoch ratio of spectrum to total,
       averaged across epochs). These are used for topomaps / plotting.

    2) `band_power_R`, `band_power_MF` (dict[band] -> (n_subj, n_ch)):
       Subject-level mean relative-band-power values, computed as the
       per-epoch ratio
            band_power[epoch] / total_power[epoch]
       averaged across epochs at the very end. This is the "mean of
       ratios" definition of relative band power; all band-level
       statistics (ROI analysis, permutation tests, ROI heatmap, line
       plots) use these values.

    The mean-of-ratios values in (2) are the canonical statistical
    measure; (1) is kept for topomap visualization of the average
    normalized spectrum.
    """
    psd_R:  np.ndarray              # (n_subjects, n_channels, n_freqs)
    psd_MF: np.ndarray              # (n_subjects, n_channels, n_freqs)
    freqs:  np.ndarray              # (n_freqs,)
    info:   mne.Info
    subject_ids: List[str] = field(default_factory=list)
    # Mean-of-ratios relative band power, per band, per (subject, channel).
    band_power_R:  Dict[str, np.ndarray] = field(default_factory=dict)
    band_power_MF: Dict[str, np.ndarray] = field(default_factory=dict)
    # Per-subject mean total power (absolute units) per channel, used for
    # the band-contribution table's denominator.
    total_R:  Optional[np.ndarray] = None      # (n_subjects, n_channels)
    total_MF: Optional[np.ndarray] = None      # (n_subjects, n_channels)

    @property
    def n_subjects(self) -> int:
        return self.psd_R.shape[0]

    @property
    def n_channels(self) -> int:
        return self.psd_R.shape[1]

    def band_power(self, band: str, condition: str = "R", relative: bool = True) -> np.ndarray:
        """Return (n_subjects, n_channels) relative band power for a given condition.

        Uses the mean-of-ratios definition: for each (subject, channel),
        the value is the across-epoch mean of
            np.trapezoid(PSD over band range) / np.trapezoid(PSD over [fmin, fmax])
        i.e. each epoch's relative contribution is computed first, then
        averaged across epochs. The `relative` argument is kept for
        backward compatibility and is ignored — these values are always
        relative.

        Fallback: if band_power_R / band_power_MF were not populated (e.g.
        the GroupPSD was built manually), integrate the stored
        subject-level mean spectrum over the band range. This recovers the
        same numerical result by linearity of np.trapezoid and np.mean.
        """
        bp = self.band_power_R if condition == "R" else self.band_power_MF
        if band in bp:
            return bp[band]
        # Fallback path for manually-constructed GroupPSDs.
        fmin, fmax = FREQUENCY_BANDS[band]
        idx = np.logical_and(self.freqs >= fmin, self.freqs <= fmax)
        psd = self.psd_R if condition == "R" else self.psd_MF
        return np.trapezoid(psd[:, :, idx], x=self.freqs[idx], axis=2)


def load_group_psd(
    file_pairs: Sequence[Tuple[str, str]],
    fmin: float = 0.5,
    fmax: float = 50.0,
    relative: bool = True,
    psd_method: str = "multitaper",
    verbose: bool = True,
) -> GroupPSD:
    """
    Load all subjects, compute PSD per epoch, and compute relative band
    power as the mean-of-ratios across epochs.

    Pipeline per subject:
      1. Read both .fif epoch files with mne.read_epochs().
      2. Compute the PSD per epoch with Epochs.compute_psd over [fmin, fmax].
         Shape: (n_epochs, n_channels, n_freqs).
      3. For each epoch and channel:
            total_power[ep, ch] = np.trapezoid(psd[ep, ch], x=freqs)
         and for each band b:
            band_power[ep, ch, b] = np.trapezoid(psd[ep, ch, band_idx],
                                                 x=freqs[band_idx])
         If `relative=True` (default), compute the per-epoch ratio
            rel[ep, ch, b] = band_power[ep, ch, b] / total_power[ep, ch]
         Otherwise rel = band_power (absolute units).
      4. Average across epochs to get one value per (subject, channel, band).
      5. The subject-level mean PSD spectrum is also stored for plotting
         (the per-epoch normalized PSD averaged across epochs when
         relative=True, otherwise the raw averaged PSD).

    Returns
    -------
    GroupPSD with:
      - psd_R / psd_MF        : (n_subj, n_ch, n_freqs)  averaged spectra
      - band_power_R / _MF    : dict[band] -> (n_subj, n_ch)  mean-of-ratios
      - total_R / total_MF    : (n_subj, n_ch)  mean absolute total power
      - freqs, info, subject_ids
    """
    psd_R_list, psd_MF_list, ids = [], [], []
    bp_R_per_band: Dict[str, List[np.ndarray]] = {b: [] for b in FREQUENCY_BANDS}
    bp_MF_per_band: Dict[str, List[np.ndarray]] = {b: [] for b in FREQUENCY_BANDS}
    total_R_list:  List[np.ndarray] = []
    total_MF_list: List[np.ndarray] = []
    info, freqs = None, None

    # Pre-compute the boolean masks once we have `freqs` (done inside the loop).
    band_idx_cache: Dict[str, np.ndarray] = {}

    for i, (f_R, f_MF) in enumerate(file_pairs):
        sid = f"S{i + 1:02d}"
        if verbose:
            print(f"[{sid}] Loading {os.path.basename(f_R)} / {os.path.basename(f_MF)}")
        try:
            ep_R  = mne.read_epochs(f_R,  preload=True, verbose="ERROR")
            ep_MF = mne.read_epochs(f_MF, preload=True, verbose="ERROR")
        except Exception as e:
            print(f"  ! skipped ({e})")
            continue

        psd_R_obj  = ep_R.compute_psd(method=psd_method,  fmin=fmin, fmax=fmax, verbose="ERROR")
        psd_MF_obj = ep_MF.compute_psd(method=psd_method, fmin=fmin, fmax=fmax, verbose="ERROR")

        psd_R,  freqs = psd_R_obj.get_data(return_freqs=True)   # (n_ep_R, n_ch, n_f)
        psd_MF, _     = psd_MF_obj.get_data(return_freqs=True)  # (n_ep_MF, n_ch, n_f)

        # Build band-frequency masks once.
        if not band_idx_cache:
            for b, (lo, hi) in FREQUENCY_BANDS.items():
                band_idx_cache[b] = (freqs >= lo) & (freqs <= hi)

        # ---- Per-epoch total power over [fmin, fmax] ----
        total_R_per_epoch  = np.trapezoid(psd_R,  x=freqs, axis=2)   # (n_ep, n_ch)
        total_MF_per_epoch = np.trapezoid(psd_MF, x=freqs, axis=2)

        # Guard against zero/negative totals (shouldn't happen for real PSDs).
        safe_total_R  = np.where(total_R_per_epoch  > 0, total_R_per_epoch,  np.nan)
        safe_total_MF = np.where(total_MF_per_epoch > 0, total_MF_per_epoch, np.nan)

        # ---- Per-epoch band power and per-epoch ratio, then mean over epochs ----
        for band, idx in band_idx_cache.items():
            band_R_per_epoch  = np.trapezoid(psd_R[:, :, idx],  x=freqs[idx], axis=2)
            band_MF_per_epoch = np.trapezoid(psd_MF[:, :, idx], x=freqs[idx], axis=2)
            if relative:
                rel_R_per_epoch  = band_R_per_epoch  / safe_total_R
                rel_MF_per_epoch = band_MF_per_epoch / safe_total_MF
            else:
                rel_R_per_epoch  = band_R_per_epoch
                rel_MF_per_epoch = band_MF_per_epoch
            # Mean across epochs -> (n_channels,)
            bp_R_per_band[band].append(np.nanmean(rel_R_per_epoch,  axis=0))
            bp_MF_per_band[band].append(np.nanmean(rel_MF_per_epoch, axis=0))

        # ---- Subject-level mean PSD spectrum (for topomap plots) ----
        if relative:
            psd_R_norm  = psd_R  / safe_total_R[:, :, np.newaxis]
            psd_MF_norm = psd_MF / safe_total_MF[:, :, np.newaxis]
            psd_R_list.append(np.nanmean(psd_R_norm,  axis=0))
            psd_MF_list.append(np.nanmean(psd_MF_norm, axis=0))
        else:
            psd_R_list.append(psd_R.mean(axis=0))
            psd_MF_list.append(psd_MF.mean(axis=0))

        # ---- Per-subject mean absolute total power (for the Sum_* table) ----
        total_R_list.append(np.nanmean(safe_total_R,  axis=0))
        total_MF_list.append(np.nanmean(safe_total_MF, axis=0))

        ids.append(sid)
        if info is None:
            info = ep_R.info

    if not psd_R_list:
        raise RuntimeError("No subjects loaded successfully.")

    band_power_R  = {b: np.stack(vs) for b, vs in bp_R_per_band.items()}
    band_power_MF = {b: np.stack(vs) for b, vs in bp_MF_per_band.items()}

    return GroupPSD(
        psd_R=np.stack(psd_R_list),
        psd_MF=np.stack(psd_MF_list),
        freqs=freqs,
        info=info,
        subject_ids=ids,
        band_power_R=band_power_R,
        band_power_MF=band_power_MF,
        total_R=np.stack(total_R_list),
        total_MF=np.stack(total_MF_list),
    )


# =============================================================================
# Statistics
# =============================================================================
def cohens_dz(x: np.ndarray, y: np.ndarray, axis: int = 0) -> np.ndarray:
    """
    Paired-samples Cohen's d_z = mean(diff) / sd(diff).
    Returns effect size with same shape as the reduced axis.
    """
    diff = x - y
    sd = np.std(diff, axis=axis, ddof=1)
    sd = np.where(sd == 0, np.nan, sd)
    return np.mean(diff, axis=axis) / sd


def permutation_per_channel(
    band_R: np.ndarray,        # (n_subj, n_ch)
    band_MF: np.ndarray,       # (n_subj, n_ch)
    n_permutations: int = 5000,
    alpha: float = 0.01,
    seed: int = 42,
) -> Dict[str, np.ndarray]:
    """Per-electrode paired permutation test on (MF - R) using
    `scipy.stats.permutation_test` with `permutation_type='samples'`
    (paired sign-flip permutation). Multiple-comparison correction
    across channels is performed with the Benjamini-Hochberg FDR procedure
    via `scipy.stats.false_discovery_control(method='bh')`."""
    n_ch = band_R.shape[1]
    p_vals = np.zeros(n_ch)

    def mean_diff(x, y):
        return np.mean(x - y)

    for ch in range(n_ch):
        res = permutation_test(
            data=(band_MF[:, ch], band_R[:, ch]),
            statistic=mean_diff,
            permutation_type="samples",
            n_resamples=n_permutations,
            alternative="two-sided",
            random_state=seed,
        )
        p_vals[ch] = res.pvalue

    p_fdr = false_discovery_control(p_vals, method="bh")
    return {
        "p":            p_vals,
        "p_fdr":        p_fdr,
        "sig":          p_vals < alpha,
        "sig_fdr":      p_fdr  < alpha,
        "dz":           cohens_dz(band_MF, band_R, axis=0),
    }


def cluster_permutation_spatial(
    band_R: np.ndarray,
    band_MF: np.ndarray,
    info: mne.Info,
    n_permutations: int = 5000,
    alpha: float = 0.01,
    seed: int = 42,
) -> Dict[str, np.ndarray]:
    """
    Spatial cluster-based permutation test on the paired difference (MF - R)
    using `mne.stats.permutation_cluster_1samp_test`. Channel adjacency is
    estimated with `mne.channels.find_ch_adjacency(ch_type='eeg')`. Two-sided
    test (`tail=0`). Returns t-statistics, cluster masks, and cluster p-values.
    """
    diff = band_MF - band_R                                  # (n_subj, n_ch)
    adjacency, _ = mne.channels.find_ch_adjacency(info, ch_type="eeg")

    t_obs, clusters, p_clusters, _ = permutation_cluster_1samp_test(
        diff,
        n_permutations=n_permutations,
        adjacency=adjacency,
        tail=0,                       # two-sided
        seed=seed,
        out_type="mask",
        verbose="ERROR",
    )

    sig_cluster = np.zeros(diff.shape[1], dtype=bool)
    for cl, p in zip(clusters, p_clusters):
        if p < alpha:
            sig_cluster |= cl

    return {
        "t_obs":         t_obs,
        "clusters":      clusters,
        "p_clusters":    p_clusters,
        "sig_cluster":   sig_cluster,
    }


def roi_analysis(
    group: GroupPSD,
    rois: Dict[str, List[str]] = ROI_DEFINITIONS,
) -> Dict[str, Dict[str, Dict[str, float]]]:
    """
    Per-ROI, per-band paired stats.
    Returns nested dict: results[band][roi] = {mean_R, mean_MF, dz, t, p, p_wilcoxon}.
    Uses `scipy.stats.ttest_rel` (parametric paired t-test) and
    `scipy.stats.wilcoxon` (non-parametric paired test).
    """
    ch_names = group.info["ch_names"]
    name_to_idx = {ch: i for i, ch in enumerate(ch_names)}

    out: Dict[str, Dict[str, Dict[str, float]]] = {}
    for band in FREQUENCY_BANDS:
        out[band] = {}
        bp_R  = group.band_power(band, "R")     # (n_subj, n_ch)
        bp_MF = group.band_power(band, "MF")

        for roi, chans in rois.items():
            idx = [name_to_idx[c] for c in chans if c in name_to_idx]
            if not idx:
                continue
            roi_R  = bp_R[:, idx].mean(axis=1)
            roi_MF = bp_MF[:, idx].mean(axis=1)

            t, p_t = ttest_rel(roi_MF, roi_R)
            try:
                _, p_w = wilcoxon(roi_MF, roi_R)
            except ValueError:
                p_w = np.nan

            out[band][roi] = {
                "n_channels": len(idx),
                "mean_R":  float(roi_R.mean()),
                "mean_MF": float(roi_MF.mean()),
                "diff":    float((roi_MF - roi_R).mean()),
                "dz":      float(cohens_dz(roi_MF, roi_R)),
                "t":       float(t),
                "p":       float(p_t),
                "p_wilcoxon": float(p_w) if not np.isnan(p_w) else np.nan,
                "subj_R":  roi_R,
                "subj_MF": roi_MF,
            }
    return out


def band_contribution_table(
    group: GroupPSD,
    aggregate: str = "mean",
) -> Dict[str, np.ndarray]:
    """
    Per-electrode table of relative-power contribution per band per state.

    Aggregates band power across subjects (default: mean across subjects)
    and returns a structured table:

      rows    = electrodes (in group.info['ch_names'] order)
      columns = one column per (band, state) plus two "Sum" columns
                (one per state) and a "Δ Sum (MF − Rest)" column.

    Per-band cells:
        For each (subject, channel), the relative band power is computed
        per epoch as
            np.trapezoid(PSD over band range) / np.trapezoid(PSD over [fmin, fmax])
        and then averaged across epochs (this is the canonical "mean of
        ratios" relative-power definition; see GroupPSD.band_power()).
        The cell value is the across-subjects mean (or median) of those
        per-(subject, channel) values.

    Sum_<state> column:
        Single np.trapezoid integral of the per-subject mean (normalized)
        PSD spectrum over the UNION of all defined band frequency ranges,
        then averaged across subjects. By construction Sum_<state> ≈ 1.0
        for every electrode when the bands cover the full normalization
        range, since the underlying spectra were normalized so that they
        integrate to 1 over [fmin, fmax] (per epoch, averaged over epochs).
        Note: this is NOT a numerical sum of the per-band columns —
        it is recomputed directly from the PSD over the merged interval.

    The returned dict has:
      'electrode'        : (n_ch,) array of channel names
      'columns'          : list of column labels in order
      'values'           : (n_ch, n_cols) float array
      'band_columns'     : list of (band, state) labels in display order
      'sum_freq_range'   : (lo, hi) of the union range used for Sum_*
      'aggregate'        : the aggregator used ('mean' or 'median')
    """
    assert aggregate in ("mean", "median")
    agg = np.nanmean if aggregate == "mean" else np.nanmedian

    bands = list(FREQUENCY_BANDS.keys())
    states = ["Rest", "MF"]
    ch_names = list(group.info["ch_names"])

    # Per-band cells (each band power is integrated with np.trapezoid in
    # GroupPSD.band_power, then averaged across subjects).
    band_columns: List[Tuple[str, str]] = []
    column_labels: List[str] = []
    cols: List[np.ndarray] = []
    for band in bands:
        for state in states:
            cond_key = "R" if state == "Rest" else "MF"
            bp = group.band_power(band, cond_key)          # (n_subj, n_ch)
            cols.append(agg(bp, axis=0))                   # (n_ch,)
            band_columns.append((band, state))
            column_labels.append(f"{band}_{state}")

    values = np.stack(cols, axis=1)                        # (n_ch, n_bands*2)

    # Per-electrode "Sum" columns: single np.trapezoid integral over the
    # union of all band ranges. We treat the union as the smallest interval
    # [lo, hi] that covers every band — this is correct as long as the bands
    # don't have gaps, which is true for the default Delta..Gamma layout.
    union_lo = min(lo for (lo, _) in FREQUENCY_BANDS.values())
    union_hi = max(hi for (_, hi) in FREQUENCY_BANDS.values())
    idx = (group.freqs >= union_lo) & (group.freqs <= union_hi)
    f_sub = group.freqs[idx]

    def sum_over_union(condition: str) -> np.ndarray:
        cond_key = "R" if condition == "Rest" else "MF"
        psd = group.psd_R if cond_key == "R" else group.psd_MF
        # Per-subject, per-channel integral over the union of band ranges.
        per_subj = np.trapezoid(psd[:, :, idx], x=f_sub, axis=2)   # (n_subj, n_ch)
        return agg(per_subj, axis=0)                                # (n_ch,)

    sum_rest = sum_over_union("Rest")[:, None]
    sum_mf   = sum_over_union("MF")[:, None]
    delta    = sum_mf - sum_rest

    values_full = np.concatenate([values, sum_rest, sum_mf, delta], axis=1)
    column_labels_full = column_labels + ["Sum_Rest", "Sum_MF", "DeltaSum_MF-Rest"]

    return {
        "electrode":      np.array(ch_names),
        "columns":        column_labels_full,
        "values":         values_full,
        "band_columns":   band_columns,
        "sum_freq_range": (float(union_lo), float(union_hi)),
        "aggregate":      aggregate,
    }


def save_band_contribution_table(
    table: Dict[str, np.ndarray],
    out_path: str,
    fmt: str = "csv",
    float_fmt: str = "%.6f",
) -> str:
    """
    Save the band-contribution table to disk.

    Parameters
    ----------
    table : dict
        Output of band_contribution_table().
    out_path : str
        Destination path. Extension is honored if it matches `fmt`.
    fmt : {'csv', 'tsv', 'markdown'}
        Output format.
    float_fmt : str
        Numeric format for the value cells (CSV/TSV only).

    Returns
    -------
    The absolute path of the file written.
    """
    assert fmt in ("csv", "tsv", "markdown")
    header = ["electrode"] + table["columns"]

    if fmt in ("csv", "tsv"):
        sep = "," if fmt == "csv" else "\t"
        lines = [sep.join(header)]
        for i, ch in enumerate(table["electrode"]):
            row = [ch] + [float_fmt % v for v in table["values"][i]]
            lines.append(sep.join(row))
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
            f.write("\n")
    else:  # markdown
        widths = [max(len(h), 10) for h in header]
        def fmt_row(cells):
            return "| " + " | ".join(c.ljust(w) for c, w in zip(cells, widths)) + " |"
        sep_row = "| " + " | ".join("-" * w for w in widths) + " |"
        lines = [fmt_row(header), sep_row]
        for i, ch in enumerate(table["electrode"]):
            row = [ch] + [float_fmt % v for v in table["values"][i]]
            lines.append(fmt_row(row))
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
            f.write("\n")

    return os.path.abspath(out_path)


# =============================================================================
# Plotting helpers
# =============================================================================
def _add_sig_markers(ax, info, mask, color="black", marker="o", size=18):
    """Add markers at significant electrode positions on a topomap axis."""
    if mask.sum() == 0:
        return
    pos = np.array([
        mne.channels.layout._find_topomap_coords(info, picks=[ch])[0]
        for ch in np.where(mask)[0]
    ])
    ax.scatter(
        pos[:, 0], pos[:, 1],
        s=size, marker=marker,
        facecolors=color, edgecolors="white", linewidths=0.8,
        zorder=10,
    )


def _topomap(data, info, ax, *, cmap="jet", vlim=(None, None),
             title=None, contours=4):
    """Thin wrapper for consistent topomap styling."""
    im, _ = mne.viz.plot_topomap(
        data, info, axes=ax, cmap=cmap, vlim=vlim,
        contours=contours, sensors=True, outlines="head",
        show=False,
    )
    if title:
        ax.set_title(title, fontsize=10, pad=4)
    return im


def _plot_overview_4rows(
    group: GroupPSD,
    stats_per_band: Dict[str, Dict[str, np.ndarray]],
    cluster_per_band: Dict[str, Dict[str, np.ndarray]],
    metric: str,                       # "diff" or "dz"
    save_path: Optional[str] = None,
    alpha_cluster: float = 0.01,
    alpha_fdr: float = 0.05,
    n_permutations: int = 5000,
):
    """
    Shared helper for the 4-row overview figures.

    Layout (rows, top -> bottom):
      1. Rest                                       (relative power, jet)
      2. Mindfulness                                (relative power, jet)
      3. Effect map (MF - Rest or Cohen's d_z)
         with FDR-significant electrodes marked
         (per-channel paired permutation test
          + Benjamini-Hochberg FDR via scipy)
      4. Same effect map with cluster-based
         significant electrodes marked
         (mne.stats.permutation_cluster_1samp_test)

    `metric`:
      - "diff" : show paired difference (MF - Rest) in rows 3 and 4.
      - "dz"   : show Cohen's d_z in rows 3 and 4.
    """
    assert metric in ("diff", "dz")
    bands = list(FREQUENCY_BANDS.keys())
    n_rows, n_cols = 4, len(bands)

    # Extra column on the right holds row-level colorbars.
    fig = plt.figure(figsize=(2.6 * n_cols + 1.8, 2.6 * n_rows + 1.0))
    gs = fig.add_gridspec(
        n_rows, n_cols + 1,
        width_ratios=[1.0] * n_cols + [0.10],
        hspace=0.45, wspace=0.20,
        left=0.06, right=0.93, top=0.88, bottom=0.09,
    )
    axes = np.empty((n_rows, n_cols), dtype=object)
    for r in range(n_rows):
        for c in range(n_cols):
            axes[r, c] = fig.add_subplot(gs[r, c])
    cax_axes = [fig.add_subplot(gs[r, n_cols]) for r in range(n_rows)]

    # ---- Titles & subtitles ----
    if metric == "diff":
        main_title = ("Group-level topographies — Rest, Mindfulness, "
                      "and paired difference (MF − Rest)")
        effect_label_short = "MF − Rest"
        effect_label_full = "Δ rel. power (MF − Rest)"
    else:
        main_title = ("Group-level topographies — Rest, Mindfulness, "
                      "and paired effect size (Cohen's $d_z$)")
        effect_label_short = "Cohen's $d_z$"
        effect_label_full = "Cohen's $d_z$"

    fig.suptitle(main_title, fontsize=14, fontweight="bold", y=0.985)
    fig.text(
        0.5, 0.945,
        f"Row 3: {effect_label_short} with electrodes significant after "
        f"per-channel paired permutation test (scipy.stats.permutation_test) "
        f"+ FDR (Benjamini-Hochberg, $\\alpha$ < {alpha_fdr}).\n"
        f"Row 4: {effect_label_short} with electrodes belonging to a "
        f"significant spatial cluster "
        f"(mne.stats.permutation_cluster_1samp_test, "
        f"{n_permutations} perms, $\\alpha$ < {alpha_cluster}).",
        ha="center", va="top", fontsize=9.5, style="italic", color="#444444",
    )

    row_titles = [
        "Rest\n(rel. power)",
        "Mindfulness\n(rel. power)",
        f"{effect_label_short}\n+ FDR sig. (p < {alpha_fdr})",
        f"{effect_label_short}\n+ cluster sig. (p < {alpha_cluster})",
    ]
    for r, t in enumerate(row_titles):
        axes[r, 0].text(-0.40, 0.5, t, transform=axes[r, 0].transAxes,
                        fontsize=10, fontweight="bold",
                        ha="right", va="center", rotation=90)

    # Per-row range bookkeeping for the right-side colorbars.
    row_ranges: List[Tuple[float, float]] = []
    row_diverging = [False, False, True, True]

    for c, band in enumerate(bands):
        bp_R  = group.band_power(band, "R").mean(axis=0)
        bp_MF = group.band_power(band, "MF").mean(axis=0)
        diff  = bp_MF - bp_R
        dz    = stats_per_band[band]["dz"]
        cl    = cluster_per_band[band]
        sig_fdr = stats_per_band[band]["sig_fdr"]

        # Effect data shown in rows 3 and 4.
        if metric == "diff":
            effect = diff
            vmax_eff = max(abs(diff.min()), abs(diff.max())) or 1e-12
        else:
            effect = dz
            vmax_eff = max(abs(np.nanmin(dz)), abs(np.nanmax(dz))) or 1e-12

        vmax_pow = max(bp_R.max(), bp_MF.max())
        vmin_pow = min(bp_R.min(), bp_MF.min())

        fmin_b, fmax_b = FREQUENCY_BANDS[band]
        axes[0, c].set_title(f"{band}\n({fmin_b}–{fmax_b} Hz)",
                             fontsize=11, fontweight="bold", pad=8)

        _topomap(bp_R,  group.info, axes[0, c], cmap="jet",
                 vlim=(vmin_pow, vmax_pow))
        _topomap(bp_MF, group.info, axes[1, c], cmap="jet",
                 vlim=(vmin_pow, vmax_pow))

        # Row 3: effect + FDR-significant electrodes.
        _topomap(effect, group.info, axes[2, c], cmap="jet",
                 vlim=(-vmax_eff, vmax_eff))
        _add_sig_markers(axes[2, c], group.info, sig_fdr,
                         color="black", marker="o", size=22)
        axes[2, c].set_title(f"{int(sig_fdr.sum())} ch FDR-sig.",
                             fontsize=9, pad=2)

        # Row 4: effect + cluster-significant electrodes.
        _topomap(effect, group.info, axes[3, c], cmap="jet",
                 vlim=(-vmax_eff, vmax_eff))
        _add_sig_markers(axes[3, c], group.info, cl["sig_cluster"],
                         color="black", marker="o", size=22)
        axes[3, c].set_title(f"{int(cl['sig_cluster'].sum())} ch in cluster",
                             fontsize=9, pad=2)

        # Per-band scale annotation under each topomap.
        axes[0, c].set_xlabel(f"[{vmin_pow:.3g}, {vmax_pow:.3g}]",
                              fontsize=7, color="#555555", labelpad=1)
        axes[1, c].set_xlabel(f"[{vmin_pow:.3g}, {vmax_pow:.3g}]",
                              fontsize=7, color="#555555", labelpad=1)
        eff_fmt = f"±{vmax_eff:.3g}" if metric == "diff" else f"±{vmax_eff:.2f}"
        axes[2, c].set_xlabel(eff_fmt, fontsize=7, color="#555555", labelpad=1)
        axes[3, c].set_xlabel(eff_fmt, fontsize=7, color="#555555", labelpad=1)

        # Per-row range bookkeeping (union across bands).
        if c == 0:
            row_ranges = [(vmin_pow, vmax_pow),
                          (vmin_pow, vmax_pow),
                          (-vmax_eff, vmax_eff),
                          (-vmax_eff, vmax_eff)]
        else:
            row_ranges[0] = (min(row_ranges[0][0], vmin_pow),
                             max(row_ranges[0][1], vmax_pow))
            row_ranges[1] = (min(row_ranges[1][0], vmin_pow),
                             max(row_ranges[1][1], vmax_pow))
            row_ranges[2] = (-max(abs(row_ranges[2][0]), vmax_eff),
                              max(abs(row_ranges[2][1]), vmax_eff))
            row_ranges[3] = (-max(abs(row_ranges[3][0]), vmax_eff),
                              max(abs(row_ranges[3][1]), vmax_eff))

    # ---- Row-level colorbars (right side) ----
    row_labels = [
        "Relative power",
        "Relative power",
        effect_label_full,
        effect_label_full,
    ]
    for cax, (vmin, vmax), label, diverging in zip(
            cax_axes, row_ranges, row_labels, row_diverging):
        if diverging:
            v = max(abs(vmin), abs(vmax))
            norm = Normalize(vmin=-v, vmax=v)
        else:
            norm = Normalize(vmin=vmin, vmax=vmax)
        sm = ScalarMappable(norm=norm, cmap="jet")
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cax)
        cb.set_label(label, fontsize=9)
        cb.ax.tick_params(labelsize=8)

    # Footnote explaining per-band scaling
    fig.text(
        0.5, 0.030,
        "Each topomap uses its own per-band color scale (printed below each "
        "topomap as [min, max] or ±max). "
        "The right-side colorbars span the union range across all bands in that row.",
        ha="center", va="bottom", fontsize=8, style="italic", color="#666666",
    )

    # Shared legend for the significance markers
    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="black",
               markeredgecolor="white", markersize=8,
               label=f"Row 3: FDR-significant electrode (BH, p < {alpha_fdr})"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="black",
               markeredgecolor="white", markersize=8,
               label=f"Row 4: electrode in significant cluster "
                     f"(p < {alpha_cluster})"),
    ]
    fig.legend(handles=legend_handles, loc="lower center",
               ncol=2, fontsize=9, bbox_to_anchor=(0.5, 0.0))

    if save_path:
        fig.savefig(save_path)
    return fig


def plot_overview_diff(
    group: GroupPSD,
    stats_per_band: Dict[str, Dict[str, np.ndarray]],
    cluster_per_band: Dict[str, Dict[str, np.ndarray]],
    save_path: Optional[str] = None,
    alpha_cluster: float = 0.01,
    alpha_fdr: float = 0.05,
    n_permutations: int = 5000,
):
    """
    4-row overview using the paired difference (MF − Rest) as the effect.

    Rows: Rest, Mindfulness, MF−Rest + FDR-significant electrodes,
    MF−Rest + cluster-significant electrodes.
    """
    return _plot_overview_4rows(
        group, stats_per_band, cluster_per_band,
        metric="diff",
        save_path=save_path,
        alpha_cluster=alpha_cluster,
        alpha_fdr=alpha_fdr,
        n_permutations=n_permutations,
    )


def plot_overview_dz(
    group: GroupPSD,
    stats_per_band: Dict[str, Dict[str, np.ndarray]],
    cluster_per_band: Dict[str, Dict[str, np.ndarray]],
    save_path: Optional[str] = None,
    alpha_cluster: float = 0.01,
    alpha_fdr: float = 0.05,
    n_permutations: int = 5000,
):
    """
    4-row overview using Cohen's d_z (paired effect size) as the effect.

    Rows: Rest, Mindfulness, Cohen's d_z + FDR-significant electrodes,
    Cohen's d_z + cluster-significant electrodes.
    """
    return _plot_overview_4rows(
        group, stats_per_band, cluster_per_band,
        metric="dz",
        save_path=save_path,
        alpha_cluster=alpha_cluster,
        alpha_fdr=alpha_fdr,
        n_permutations=n_permutations,
    )


def plot_band_summary(
    group: GroupPSD,
    band: str,
    stats: Dict[str, np.ndarray],
    cluster: Dict[str, np.ndarray],
    roi_results: Dict[str, Dict[str, float]],
    save_path: Optional[str] = None,
    alpha_cluster: float = 0.01,
    n_permutations: int = 5000,
):
    """
    Publication-ready single-band figure:
      [Rest topo] [MF topo] [Diff + cluster] [Cohen's d]
      [ROI bar plot with subject lines]
    """
    bp_R  = group.band_power(band, "R")
    bp_MF = group.band_power(band, "MF")
    mean_R, mean_MF = bp_R.mean(0), bp_MF.mean(0)
    diff = mean_MF - mean_R
    dz = stats["dz"]

    fmin_b, fmax_b = FREQUENCY_BANDS[band]

    fig = plt.figure(figsize=(13, 7.5), constrained_layout=True)
    gs = fig.add_gridspec(2, 4, height_ratios=[1.0, 0.95])

    fig.suptitle(
        f"{band} band ({fmin_b}–{fmax_b} Hz) — Rest vs. Mindfulness "
        f"(N = {group.n_subjects})",
        fontsize=14, fontweight="bold",
    )

    vmax_pow = max(mean_R.max(), mean_MF.max())
    vmin_pow = min(mean_R.min(), mean_MF.min())
    vmax_d   = max(abs(diff.min()), abs(diff.max())) or 1e-12
    vmax_dz  = max(abs(np.nanmin(dz)), abs(np.nanmax(dz))) or 1e-12

    ax_R   = fig.add_subplot(gs[0, 0])
    ax_MF  = fig.add_subplot(gs[0, 1])
    ax_d   = fig.add_subplot(gs[0, 2])
    ax_dz  = fig.add_subplot(gs[0, 3])

    im0 = _topomap(mean_R,  group.info, ax_R,  cmap="jet",
                   vlim=(vmin_pow, vmax_pow),
                   title="Rest — mean rel. power")
    im1 = _topomap(mean_MF, group.info, ax_MF, cmap="jet",
                   vlim=(vmin_pow, vmax_pow),
                   title="Mindfulness — mean rel. power")
    im2 = _topomap(diff, group.info, ax_d, cmap="jet",
                   vlim=(-vmax_d, vmax_d),
                   title=f"MF − Rest\n(black • : cluster p < {alpha_cluster})")
    _add_sig_markers(ax_d, group.info, cluster["sig_cluster"],
                     color="black", marker="o", size=24)
    im3 = _topomap(dz, group.info, ax_dz, cmap="jet",
                   vlim=(-vmax_dz, vmax_dz),
                   title=f"Cohen's $d_z$ (paired)\n"
                         f"(black • : cluster p < {alpha_cluster})")
    _add_sig_markers(ax_dz, group.info, cluster["sig_cluster"],
                     color="black", marker="o", size=24)

    # Compact colorbars
    fig.colorbar(im0, ax=[ax_R, ax_MF], shrink=0.7, location="bottom",
                 pad=0.02, label="Relative power")
    fig.colorbar(im2, ax=ax_d,  shrink=0.7, location="bottom",
                 pad=0.02, label="Δ relative power (MF − Rest)")
    fig.colorbar(im3, ax=ax_dz, shrink=0.7, location="bottom",
                 pad=0.02, label="Cohen's $d_z$")

    # ----- Bottom: ROI bar plot with individual subject lines -----
    ax_roi = fig.add_subplot(gs[1, :])
    rois = list(roi_results.keys())
    x = np.arange(len(rois))
    width = 0.35

    means_R  = [roi_results[r]["mean_R"]  for r in rois]
    means_MF = [roi_results[r]["mean_MF"] for r in rois]
    sems_R   = [roi_results[r]["subj_R"].std(ddof=1)  / np.sqrt(group.n_subjects)
                for r in rois]
    sems_MF  = [roi_results[r]["subj_MF"].std(ddof=1) / np.sqrt(group.n_subjects)
                for r in rois]

    ax_roi.bar(x - width/2, means_R,  width, yerr=sems_R,
               label="Rest",        color="#5B8FB9", alpha=0.85, capsize=3)
    ax_roi.bar(x + width/2, means_MF, width, yerr=sems_MF,
               label="Mindfulness", color="#B6464A", alpha=0.85, capsize=3)

    # Individual-subject lines
    line_handle = None
    for i, r in enumerate(rois):
        rR  = roi_results[r]["subj_R"]
        rMF = roi_results[r]["subj_MF"]
        for s in range(len(rR)):
            ln, = ax_roi.plot([x[i] - width/2, x[i] + width/2],
                              [rR[s], rMF[s]],
                              color="gray", alpha=0.25, linewidth=0.6,
                              marker="o", markersize=2.5, zorder=2)
            if line_handle is None:
                line_handle = ln
    if line_handle is not None:
        line_handle.set_label("Individual subjects")

    # Significance annotations
    ymax_overall = max(max(means_R), max(means_MF)) * 1.15
    for i, r in enumerate(rois):
        p = roi_results[r]["p"]
        dz_ = roi_results[r]["dz"]
        if p < 0.001:   stars = "***"
        elif p < 0.01:  stars = "**"
        elif p < 0.05:  stars = "*"
        else:           stars = "n.s."
        ax_roi.text(x[i], ymax_overall * 0.98,
                    f"{stars}\n$d_z$={dz_:+.2f}",
                    ha="center", va="top", fontsize=9)

    ax_roi.set_xticks(x)
    ax_roi.set_xticklabels(rois)
    ax_roi.set_ylabel("Mean relative power")
    ax_roi.set_xlabel("Region of interest (ROI)")
    ax_roi.set_title(
        "ROI analysis — paired t-test (scipy.stats.ttest_rel), "
        "mean ± SEM\n"
        "* p < 0.05    ** p < 0.01    *** p < 0.001    n.s. not significant",
        fontsize=10,
    )
    ax_roi.legend(loc="upper right")
    ax_roi.set_ylim(0, ymax_overall * 1.05)

    if save_path:
        fig.savefig(save_path)
    return fig


def plot_subject_variability(
    group: GroupPSD,
    band: str,
    save_path: Optional[str] = None,
):
    """
    Per-subject variability for a single band:
      - Left: paired strip/line plot (each subject = one line, R vs MF, averaged across all electrodes)
      - Right: per-subject difference (MF - R) topomap grid
    """
    bp_R  = group.band_power(band, "R").mean(axis=1)   # mean across channels per subj
    bp_MF = group.band_power(band, "MF").mean(axis=1)
    diff_per_subj = group.band_power(band, "MF") - group.band_power(band, "R")  # (n_subj, n_ch)

    n_subj = group.n_subjects
    n_cols_topo = 6
    n_rows_topo = int(np.ceil(n_subj / n_cols_topo))

    fmin_b, fmax_b = FREQUENCY_BANDS[band]

    fig = plt.figure(figsize=(15, 3 + 1.5 * n_rows_topo), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 2.0])

    fig.suptitle(
        f"Per-subject variability — {band} band ({fmin_b}–{fmax_b} Hz)",
        fontsize=14, fontweight="bold",
    )

    # --- Left: paired plot ---
    ax_pair = fig.add_subplot(gs[0, 0])
    up_handle = down_handle = None
    for s in range(n_subj):
        if bp_MF[s] > bp_R[s]:
            color, label = "#B6464A", "MF > Rest"
        else:
            color, label = "#5B8FB9", "MF < Rest"
        ln, = ax_pair.plot([0, 1], [bp_R[s], bp_MF[s]],
                           color=color, alpha=0.55, marker="o", markersize=4)
        if label == "MF > Rest" and up_handle is None:
            up_handle = ln; up_handle.set_label("MF > Rest")
        elif label == "MF < Rest" and down_handle is None:
            down_handle = ln; down_handle.set_label("MF < Rest")
    # Group mean overlay
    ax_pair.plot([0, 1], [bp_R.mean(), bp_MF.mean()],
                 color="black", marker="s", markersize=8,
                 linewidth=2.5, label="Group mean", zorder=10)
    ax_pair.set_xticks([0, 1])
    ax_pair.set_xticklabels(["Rest", "Mindfulness"])
    ax_pair.set_ylabel("Mean relative power\n(averaged across all electrodes)")
    ax_pair.set_title(f"Subject-level change\n(N = {n_subj} subjects)")
    ax_pair.legend(loc="best", fontsize=8)
    ax_pair.set_xlim(-0.3, 1.3)

    # --- Right: per-subject diff topomaps ---
    gs_right = gs[0, 1].subgridspec(n_rows_topo, n_cols_topo, hspace=0.3, wspace=0.1)
    vmax = np.nanpercentile(np.abs(diff_per_subj), 95) or 1e-12

    last_im = None
    for s in range(n_subj):
        r, c = divmod(s, n_cols_topo)
        ax = fig.add_subplot(gs_right[r, c])
        last_im = _topomap(diff_per_subj[s], group.info, ax,
                           cmap="RdBu_r", vlim=(-vmax, vmax),
                           title=group.subject_ids[s])

    # Hide unused
    for s in range(n_subj, n_rows_topo * n_cols_topo):
        r, c = divmod(s, n_cols_topo)
        ax = fig.add_subplot(gs_right[r, c])
        ax.axis("off")

    # Add subtitle for the right side
    fig.text(0.68, 0.96,
             "Per-subject MF − Rest topographies (color scale: ±95th percentile)",
             ha="center", va="top", fontsize=10, style="italic", color="#444444")

    if save_path:
        fig.savefig(save_path)
    return fig


def plot_roi_heatmap(roi_all: Dict[str, Dict[str, Dict[str, float]]],
                     save_path: Optional[str] = None):
    """Heatmap: rows = bands, cols = ROIs, cell = Cohen's d_z, asterisks = p < .05."""
    bands = list(roi_all.keys())
    rois = list(next(iter(roi_all.values())).keys())

    dz_mat = np.array([[roi_all[b][r]["dz"] for r in rois] for b in bands])
    p_mat  = np.array([[roi_all[b][r]["p"]  for r in rois] for b in bands])

    fig, ax = plt.subplots(figsize=(1.4 * len(rois) + 2.5, 0.8 * len(bands) + 2.0))
    vmax = np.nanmax(np.abs(dz_mat)) or 1e-12
    im = ax.imshow(dz_mat, cmap="jet", vmin=-vmax, vmax=vmax, aspect="auto")

    ax.set_xticks(range(len(rois)))
    ax.set_xticklabels(rois)
    ax.set_yticks(range(len(bands)))
    ax.set_yticklabels(bands)
    ax.set_xlabel("Region of interest (ROI)")
    ax.set_ylabel("Frequency band")
    ax.set_title("ROI × Band paired effect sizes (Cohen's $d_z$, MF − Rest)\n"
                 "Asterisks: paired t-test (scipy.stats.ttest_rel) — "
                 "* p < 0.05    ** p < 0.01    *** p < 0.001",
                 fontweight="bold", fontsize=11)

    # Annotate
    for i in range(len(bands)):
        for j in range(len(rois)):
            p = p_mat[i, j]
            if p < 0.001:   star = "***"
            elif p < 0.01:  star = "**"
            elif p < 0.05:  star = "*"
            else:           star = ""
            ax.text(j, i, f"{dz_mat[i, j]:+.2f}\n{star}",
                    ha="center", va="center", fontsize=9,
                    color="white" if abs(dz_mat[i, j]) > vmax * 0.6 else "black")

    fig.colorbar(im, ax=ax, label="Cohen's $d_z$ (MF − Rest)", shrink=0.8)
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_band_change_lines(
    group: GroupPSD,
    roi_all: Dict[str, Dict[str, Dict[str, float]]],
    save_path: Optional[str] = None,
):
    """
    Average change (MF − Rest) per band as a line plot.

    Two side-by-side panels:
      Left  — One line per frequency band; x-axis = ROIs, y-axis = mean
              (MF − Rest) ± SEM across subjects. Shows the spatial pattern
              of the average change for each band.
      Right — One line per frequency band; x-axis = bands (single point),
              y-axis = mean (MF − Rest) ± 95 % CI across subjects, computed
              from the all-electrode mean. Shows the global magnitude of
              the change per band, irrespective of topography.

    Significance markers (paired t-test on each ROI x band cell from
    `roi_all`, scipy.stats.ttest_rel) are drawn on the left panel.
    """
    bands = list(FREQUENCY_BANDS.keys())
    rois = list(next(iter(roi_all.values())).keys())

    fig, (ax_l, ax_r) = plt.subplots(
        1, 2, figsize=(13, 5.8),
        gridspec_kw=dict(width_ratios=[1.7, 1.0], wspace=0.25),
    )

    fig.suptitle("Average change (Mindfulness − Rest) across frequency bands",
                 fontsize=14, fontweight="bold", y=0.985)
    fig.text(
        0.5, 0.935,
        f"Mean across N = {group.n_subjects} subjects. "
        f"Left: per-ROI change (mean ± SEM). "
        f"Right: global change averaged across all electrodes (mean ± 95 % CI).",
        ha="center", va="top", fontsize=10, style="italic", color="#444444",
    )

    # Distinct color per band (qualitative palette)
    cmap = plt.get_cmap("tab10")
    band_colors = {b: cmap(i % 10) for i, b in enumerate(bands)}

    # ---------- LEFT PANEL: per-ROI change, one line per band ----------
    x = np.arange(len(rois))
    y_max_abs = 0.0
    for b in bands:
        diffs = np.array([
            roi_all[b][r]["subj_MF"] - roi_all[b][r]["subj_R"]
            for r in rois
        ])  # (n_rois, n_subj)
        means = diffs.mean(axis=1)
        sems = diffs.std(axis=1, ddof=1) / np.sqrt(group.n_subjects)
        y_max_abs = max(y_max_abs, np.max(np.abs(means) + sems))

        ax_l.errorbar(
            x, means, yerr=sems,
            label=f"{b} ({FREQUENCY_BANDS[b][0]}–{FREQUENCY_BANDS[b][1]} Hz)",
            color=band_colors[b], marker="o", markersize=6,
            linewidth=2, capsize=3, alpha=0.9,
        )

        # Significance stars per ROI (above the marker)
        for i, r in enumerate(rois):
            p = roi_all[b][r]["p"]
            if p < 0.001:   star = "***"
            elif p < 0.01:  star = "**"
            elif p < 0.05:  star = "*"
            else:           continue
            ax_l.text(x[i], means[i] + sems[i] + y_max_abs * 0.04,
                      star, ha="center", va="bottom",
                      fontsize=9, color=band_colors[b], fontweight="bold")

    ax_l.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
    ax_l.set_xticks(x)
    ax_l.set_xticklabels(rois)
    ax_l.set_xlabel("Region of interest (ROI)")
    ax_l.set_ylabel("Mean change in relative power\n(MF − Rest)")
    ax_l.set_title(
        "Per-ROI average change (mean ± SEM)\n"
        "Stars: paired t-test (scipy.stats.ttest_rel) — "
        "* p < 0.05  ** p < 0.01  *** p < 0.001",
        fontsize=10,
    )
    ax_l.legend(loc="best", fontsize=8, ncol=2, title="Frequency band")
    ax_l.set_ylim(-y_max_abs * 1.25, y_max_abs * 1.30)

    # ---------- RIGHT PANEL: global change, one point per band ----------
    band_means: List[float] = []
    band_ci_lo: List[float] = []
    band_ci_hi: List[float] = []
    for b in bands:
        # Mean across all electrodes per subject, then MF - Rest per subject.
        diff_subj = (group.band_power(b, "MF").mean(axis=1)
                     - group.band_power(b, "R").mean(axis=1))
        m = diff_subj.mean()
        sem = diff_subj.std(ddof=1) / np.sqrt(group.n_subjects)
        ci = 1.96 * sem  # ~95 % CI (normal approx; OK for N >> 10)
        band_means.append(m)
        band_ci_lo.append(m - ci)
        band_ci_hi.append(m + ci)

    xb = np.arange(len(bands))
    # Connecting line for visual continuity across bands
    ax_r.plot(xb, band_means, color="#444444", linewidth=1.2,
              alpha=0.4, zorder=1)
    # Per-band points + error bars in the band's color
    for i, b in enumerate(bands):
        ax_r.errorbar(
            xb[i], band_means[i],
            yerr=[[band_means[i] - band_ci_lo[i]],
                  [band_ci_hi[i] - band_means[i]]],
            fmt="o", color=band_colors[b], markersize=8,
            linewidth=2, capsize=4, zorder=3,
        )

    ax_r.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
    ax_r.set_xticks(xb)
    ax_r.set_xticklabels(bands, rotation=20, ha="right")
    ax_r.set_xlabel("Frequency band")
    ax_r.set_ylabel("Mean change in relative power\n(MF − Rest, all electrodes)")
    ax_r.set_title("Global average change per band\n(mean ± 95 % CI)",
                   fontsize=10)

    # Manual layout: leave room at the top for suptitle + subtitle.
    fig.subplots_adjust(left=0.07, right=0.98, bottom=0.09, top=0.78,
                        wspace=0.30)

    if save_path:
        fig.savefig(save_path)
    return fig


# =============================================================================
# Top-level driver
# =============================================================================
def run_full_analysis(
    file_pairs: Sequence[Tuple[str, str]],
    out_dir: str = "figures",
    fmin: float = 0.5,
    fmax: float = 50.0,
    n_permutations: int = 5000,
    alpha: float = 0.01,
    relative: bool = True,
    psd_method: str = "multitaper",
):
    os.makedirs(out_dir, exist_ok=True)
    logger = MethodsLogger()
    logger.set_header(
        out_dir=os.path.abspath(out_dir),
        fmin_Hz=fmin,
        fmax_Hz=fmax,
        psd_method=psd_method,
        relative_power=relative,
        n_permutations=n_permutations,
        alpha=alpha,
        n_file_pairs=len(file_pairs),
        frequency_bands=", ".join(f"{b} ({lo}-{hi} Hz)"
                                  for b, (lo, hi) in FREQUENCY_BANDS.items()),
        ROIs=", ".join(f"{r}({len(ch)} ch)" for r, ch in ROI_DEFINITIONS.items()),
    )

    # -------------------------------------------------------------------------
    print("=" * 70)
    print("STEP 1/4: Loading data and computing PSDs")
    print("=" * 70)
    group = load_group_psd(file_pairs, fmin=fmin, fmax=fmax,
                           relative=relative, psd_method=psd_method)
    print(f"  -> Loaded {group.n_subjects} subjects, {group.n_channels} channels, "
          f"{len(group.freqs)} freqs ({group.freqs[0]:.1f}–{group.freqs[-1]:.1f} Hz)")

    logger.add(
        "STEP 1 — Data loading & PSD computation",
        [
            f"Function:           load_group_psd()",
            f"Subjects loaded:    {group.n_subjects} / {len(file_pairs)} requested",
            f"Channels:           {group.n_channels}",
            f"Frequency range:    {group.freqs[0]:.2f}–{group.freqs[-1]:.2f} Hz "
            f"({len(group.freqs)} bins)",
            "",
            "Pipeline per subject (mean-of-ratios relative power):",
            "  1. Epochs are read with mne.read_epochs() for both Rest (_R) "
            "and Mindfulness (_MF) recordings.",
            f"  2. PSD is computed per epoch with "
            f"mne.Epochs.compute_psd(method='{psd_method}', "
            f"fmin={fmin}, fmax={fmax}).",
            "     Shape: (n_epochs, n_channels, n_freqs).",
            "",
            "  3. For each (epoch, channel):",
            "       total_power[ep, ch] = np.trapezoid(psd[ep, ch], x=freqs)",
            "     For each band b:",
            "       band_power[ep, ch, b] = "
            "np.trapezoid(psd[ep, ch, band_idx], x=freqs[band_idx])",
            f"  4. {'Relative power' if relative else 'Absolute power'}: "
            + ("the per-epoch RATIO is computed first, then averaged across epochs:"
               if relative else
               "values are kept in absolute units (V^2/Hz)."),
            *(["       rel[ep, ch, b] = band_power[ep, ch, b] / "
               "total_power[ep, ch]",
               "       relative_band_power[subj, ch, b] = "
               "np.nanmean(rel[:, ch, b], axis=0)",
               "     This is the canonical 'mean of ratios' definition. "
               "Zero/negative totals",
               "     are converted to NaN (np.where(total > 0, total, nan)) "
               "and excluded by",
               "     numpy.nanmean."]
              if relative else []),
            "",
            "  5. For plotting only, the subject-level mean PSD spectrum is "
            "also stored:",
            "       psd_norm[ep] = psd[ep] / total[ep]    "
            "(per-epoch normalized spectrum)",
            "       group.psd_<state>[subj] = "
            "np.nanmean(psd_norm, axis=epochs)",
            "     This averaged-normalized spectrum is used for topomap "
            "displays of the",
            "     average PSD shape. Band-level statistics never use it — "
            "they always",
            "     use the per-epoch ratios from step 4.",
            "",
            "GroupPSD.band_power(band, state) returns the (n_subj, n_ch) "
            "mean-of-ratios",
            "values from step 4, NOT a re-integration of the averaged "
            "normalized spectrum.",
        ],
    )

    # -------------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("STEP 2/4: Per-channel + cluster-based permutation tests")
    print("=" * 70)
    stats_per_band: Dict[str, Dict[str, np.ndarray]] = {}
    cluster_per_band: Dict[str, Dict[str, np.ndarray]] = {}
    band_stat_lines: List[str] = []
    for band in FREQUENCY_BANDS:
        bp_R  = group.band_power(band, "R")
        bp_MF = group.band_power(band, "MF")

        stats_per_band[band]   = permutation_per_channel(
            bp_R, bp_MF, n_permutations=n_permutations, alpha=alpha)
        cluster_per_band[band] = cluster_permutation_spatial(
            bp_R, bp_MF, group.info, n_permutations=n_permutations, alpha=alpha)

        s = stats_per_band[band]
        c = cluster_per_band[band]
        line = (f"  {band:9s}  uncorr={s['sig'].sum():2d}  "
                f"FDR={s['sig_fdr'].sum():2d}  "
                f"cluster={c['sig_cluster'].sum():2d}  "
                f"|dz|max={np.nanmax(np.abs(s['dz'])):.2f}")
        print(line)
        band_stat_lines.append(line)

    logger.add(
        "STEP 2 — Statistical testing (per channel & cluster-based)",
        [
            "Per-channel paired permutation test",
            "-" * 78,
            "  Function:         permutation_per_channel()",
            "  Backend:          scipy.stats.permutation_test",
            "                    (permutation_type='samples', "
            "alternative='two-sided',",
            f"                     n_resamples={n_permutations}, "
            f"random_state=42)",
            "  Test statistic:   mean(MF - Rest)  (paired difference)",
            "  Effect size:      Cohen's d_z = mean(diff) / sd(diff)  "
            "(cohens_dz())",
            "",
            "Multiple-comparison correction across channels",
            "-" * 78,
            "  Function:         scipy.stats.false_discovery_control(method='bh')",
            "  Procedure:        Benjamini-Hochberg FDR control",
            f"  Significance:     uncorrected & FDR-corrected at "
            f"alpha = {alpha}",
            "",
            "Spatial cluster-based permutation test",
            "-" * 78,
            "  Function:         cluster_permutation_spatial()",
            "  Backend:          mne.stats.permutation_cluster_1samp_test",
            f"                    (n_permutations={n_permutations}, "
            "tail=0, out_type='mask',",
            "                     seed=42)",
            "  Adjacency:        mne.channels.find_ch_adjacency(ch_type='eeg')",
            f"  Significance:     cluster p-value < {alpha} "
            "(two-sided, default cluster-forming threshold)",
            "",
            "Per-band summary (significant channels):",
        ] + band_stat_lines,
    )

    # -------------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("STEP 3/4: ROI analysis")
    print("=" * 70)
    roi_all = roi_analysis(group)
    roi_lines: List[str] = []
    for band, rois in roi_all.items():
        for r, st in rois.items():
            sig = "*" if st["p"] < 0.05 else " "
            line = (f"  {sig} {band:9s} {r:9s}  "
                    f"R={st['mean_R']:.4f}  MF={st['mean_MF']:.4f}  "
                    f"dz={st['dz']:+.3f}  t={st['t']:+.2f}  "
                    f"p={st['p']:.4f}  p_wilcoxon={st['p_wilcoxon']:.4f}")
            print(line)
            roi_lines.append(line)

    logger.add(
        "STEP 3 — ROI analysis",
        [
            "Function:           roi_analysis()",
            "ROI definitions (10-20 system):",
            *[f"  {r:10s} -> {', '.join(ch)}" for r, ch in ROI_DEFINITIONS.items()],
            "",
            "For each band x ROI:",
            "  1. Band power per channel is averaged across the ROI channels "
            "(per subject).",
            "  2. Paired parametric test: scipy.stats.ttest_rel(MF, Rest).",
            "  3. Paired non-parametric test: scipy.stats.wilcoxon(MF, Rest).",
            "  4. Effect size: Cohen's d_z = mean(diff) / sd(diff).",
            "  5. SEM computed for plotting: std(condition, ddof=1) / sqrt(N).",
            "",
            "Per-band, per-ROI results "
            "('*' marks p < 0.05 in the parametric t-test):",
        ] + roi_lines,
    )

    # -------------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("STEP 3b/4: Per-electrode band-contribution table")
    print("=" * 70)
    contrib_table = band_contribution_table(group, aggregate="mean")
    contrib_csv = os.path.join(out_dir, "band_contributions_per_electrode.csv")
    contrib_md  = os.path.join(out_dir, "band_contributions_per_electrode.md")
    save_band_contribution_table(contrib_table, contrib_csv, fmt="csv")
    save_band_contribution_table(contrib_table, contrib_md,  fmt="markdown")
    print(f"  Saved: {contrib_csv}")
    print(f"  Saved: {contrib_md}")

    # Compact summary on stdout: per-electrode Sum_Rest, Sum_MF, Δ
    sum_rest_col = contrib_table["columns"].index("Sum_Rest")
    sum_mf_col   = contrib_table["columns"].index("Sum_MF")
    delta_col    = contrib_table["columns"].index("DeltaSum_MF-Rest")
    summary_lines: List[str] = []
    summary_lines.append("  Electrode-level sum of band contributions "
                         "(mean across subjects):")
    summary_lines.append(f"  {'Electrode':<10s}  {'Sum_Rest':>10s}  "
                         f"{'Sum_MF':>10s}  {'Δ (MF-Rest)':>14s}")
    for i, ch in enumerate(contrib_table["electrode"]):
        v = contrib_table["values"][i]
        line = (f"  {ch:<10s}  {v[sum_rest_col]:10.4f}  "
                f"{v[sum_mf_col]:10.4f}  {v[delta_col]:+14.5f}")
        print(line)
        summary_lines.append(line)

    logger.add(
        "STEP 3b — Per-electrode band-contribution table",
        [
            "Function:           band_contribution_table() + "
            "save_band_contribution_table().",
            "Aggregator:         numpy.nanmean across subjects "
            "(set aggregate='median' for nanmedian).",
            "Output files:",
            f"  {contrib_csv}",
            f"  {contrib_md}",
            "",
            "Table layout:",
            "  rows    = electrodes (in group.info['ch_names'] order).",
            "  columns = one column per (band, state), in display order",
            "            Delta_Rest, Delta_MF, Theta_Rest, Theta_MF, ..., Gamma_MF.",
            "            Each cell is the across-subjects mean of the band's "
            "relative-power",
            "            contribution at that electrode, computed as a "
            "numpy.trapezoid integral",
            "            of the (relative-normalized) PSD over the band's "
            "frequency range",
            "            (GroupPSD.band_power()).",
            "  + Sum_Rest:  single numpy.trapezoid integral of the PSD over the "
            "UNION of all",
            "               defined band ranges (Rest condition). NOT a numerical "
            "sum of the",
            "               per-band columns — recomputed directly from the PSD "
            "over the merged",
            f"               interval [{contrib_table['sum_freq_range'][0]:.2f}, "
            f"{contrib_table['sum_freq_range'][1]:.2f}] Hz.",
            "  + Sum_MF:    same, for the Mindfulness condition.",
            "  + DeltaSum_MF-Rest: Sum_MF - Sum_Rest.",
            "",
            "Because the stored PSDs were normalized to relative power over "
            "[fmin, fmax]",
            "by load_group_psd(relative=True), each Sum_<state> equals the "
            "fraction of total",
            "power that falls within the union of the defined band ranges. "
            "With the default",
            "bands (Delta 0.5-4 Hz through Gamma 30-50 Hz) covering nearly the "
            "full",
            "normalization range, Sum_* is close to 1.0 for every electrode "
            "and Δ ≈ 0.",
            "",
        ] + summary_lines,
    )

    # -------------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("STEP 4/4: Generating figures")
    print("=" * 70)
    fig_overview_diff = os.path.join(out_dir, "00a_overview_diff.png")
    plot_overview_diff(
        group, stats_per_band, cluster_per_band,
        save_path=fig_overview_diff,
        alpha_cluster=alpha,
        alpha_fdr=alpha,
        n_permutations=n_permutations,
    )
    logger.add(
        "FIGURE 00a_overview_diff.png — Group topographies "
        "(effect = paired difference)",
        [
            f"File:               {fig_overview_diff}",
            "Generator:          plot_overview_diff() "
            "(wraps _plot_overview_4rows() with metric='diff')",
            "Layout:             4 rows x N_bands columns, "
            "with one shared colorbar per row on the right.",
            "Rows (top -> bottom):",
            "  Row 1 — Rest:        group-mean relative band power per electrode.",
            "                        Plotted with mne.viz.plot_topomap (cmap='jet').",
            "                        Rest and MF share the same color scale per band.",
            "  Row 2 — Mindfulness: same as Row 1 for the MF condition.",
            "  Row 3 — MF - Rest with FDR-significant electrodes:",
            "                        Group-mean paired difference. "
            "Black markers at electrodes",
            "                        significant after per-channel paired "
            "permutation test",
            "                        (scipy.stats.permutation_test, "
            "permutation_type='samples',",
            f"                         n_resamples={n_permutations}, "
            "two-sided) followed by",
            "                        Benjamini-Hochberg FDR correction "
            "across channels",
            f"                        (scipy.stats.false_discovery_control"
            f"(method='bh'), alpha < {alpha}).",
            "  Row 4 — MF - Rest with cluster-significant electrodes:",
            "                        Group-mean paired difference. "
            "Black markers at electrodes",
            "                        belonging to a significant spatial "
            "cluster from",
            "                        mne.stats.permutation_cluster_1samp_test",
            f"                        (n_permutations={n_permutations}, "
            f"two-sided, p < {alpha}).",
            "                        Channel adjacency from "
            "mne.channels.find_ch_adjacency.",
            "Colorbars:          One row-level colorbar on the right per row "
            "(power, power,",
            "                    Δ rel. power, Δ rel. power). Each colorbar "
            "spans the union range",
            "                    across all bands in that row. Per-band "
            "ranges are annotated under",
            "                    each topomap ([min, max] for power rows, "
            "±max for diverging rows).",
            "Legend:             Two markers explaining the FDR row vs. the "
            "cluster row.",
            "Subtitle:           summarizes the two statistical procedures "
            "and their parameters.",
        ],
    )

    fig_overview_dz = os.path.join(out_dir, "00b_overview_dz.png")
    plot_overview_dz(
        group, stats_per_band, cluster_per_band,
        save_path=fig_overview_dz,
        alpha_cluster=alpha,
        alpha_fdr=alpha,
        n_permutations=n_permutations,
    )
    logger.add(
        "FIGURE 00b_overview_dz.png — Group topographies "
        "(effect = Cohen's d_z)",
        [
            f"File:               {fig_overview_dz}",
            "Generator:          plot_overview_dz() "
            "(wraps _plot_overview_4rows() with metric='dz')",
            "Layout:             Identical to 00a_overview_diff.png, but "
            "rows 3 and 4 show",
            "                    the per-channel paired effect size "
            "(Cohen's d_z) instead of",
            "                    the raw MF - Rest difference.",
            "Rows (top -> bottom):",
            "  Row 1 — Rest:        group-mean relative band power per electrode.",
            "  Row 2 — Mindfulness: same for the MF condition.",
            "  Row 3 — Cohen's d_z with FDR-significant electrodes:",
            "                        d_z = mean(MF - Rest) / sd(MF - Rest) "
            "per channel",
            "                        (cohens_dz()). Black markers at channels "
            "significant",
            "                        after per-channel paired permutation "
            "(scipy.stats.permutation_test)",
            "                        + Benjamini-Hochberg FDR "
            "(scipy.stats.false_discovery_control,",
            f"                         method='bh', alpha < {alpha}).",
            "  Row 4 — Cohen's d_z with cluster-significant electrodes:",
            "                        Same d_z map. Black markers at electrodes "
            "belonging to a",
            "                        significant spatial cluster from "
            "mne.stats.permutation_cluster_1samp_test",
            f"                        (n_permutations={n_permutations}, "
            f"two-sided, p < {alpha},",
            "                         adjacency from "
            "mne.channels.find_ch_adjacency).",
            "Colorbars:          Power / power / Cohen's d_z / Cohen's d_z, "
            "right-side, per row.",
            "Note:               The cluster test is performed on the "
            "paired difference",
            "                    (MF - Rest); its significant electrodes are "
            "overlaid on the",
            "                    d_z map here because d_z and the difference "
            "share the sign",
            "                    and electrode order; only the per-channel "
            "scaling differs.",
            "Legend:             Two markers explaining the FDR row vs. the "
            "cluster row.",
        ],
    )

    fig_heatmap = os.path.join(out_dir, "01_roi_heatmap.png")
    plot_roi_heatmap(roi_all, save_path=fig_heatmap)
    logger.add(
        "FIGURE 01_roi_heatmap.png — ROI x Band effect-size heatmap",
        [
            f"File:               {fig_heatmap}",
            "Generator:          plot_roi_heatmap()",
            "Layout:             Heatmap with rows = frequency bands, "
            "cols = ROIs.",
            "Cell value:         Cohen's d_z for the paired MF - Rest "
            "difference (cohens_dz()).",
            "Colormap:           'jet', symmetric around 0 "
            "(vmin=-|max|, vmax=+|max|).",
            "Cell annotations:   numeric d_z plus significance stars from",
            "                    scipy.stats.ttest_rel:",
            "                    * p < 0.05, ** p < 0.01, *** p < 0.001.",
            "Axes labels:        x = 'Region of interest (ROI)', "
            "y = 'Frequency band'.",
        ],
    )

    fig_change = os.path.join(out_dir, "01b_band_change_lines.png")
    plot_band_change_lines(group, roi_all, save_path=fig_change)
    logger.add(
        "FIGURE 01b_band_change_lines.png — Average change per band (line plot)",
        [
            f"File:               {fig_change}",
            "Generator:          plot_band_change_lines()",
            "Purpose:            Compact summary of the average change "
            "(MF - Rest) per frequency band.",
            "Layout:             Two side-by-side panels.",
            "",
            "Left panel — Per-ROI change, one line per band:",
            "  - x-axis:         5 ROIs (Frontal, Central, Parietal, "
            "Occipital, Temporal).",
            "  - y-axis:         Mean change in relative power (MF - Rest), "
            "averaged across",
            "                    subjects.",
            "  - Lines:          One line per frequency band, each in a "
            "distinct color from",
            "                    matplotlib's 'tab10' palette. "
            "Markers are circles.",
            "  - Error bars:     SEM across subjects "
            "(std with ddof=1 / sqrt(N)).",
            "  - Reference:      Dashed horizontal line at y = 0 marks "
            "no change.",
            "  - Significance:   Stars above each marker from the paired "
            "t-test stored in",
            "                    `roi_all` (scipy.stats.ttest_rel): "
            "* p < 0.05, ** p < 0.01,",
            "                    *** p < 0.001. Star color matches the "
            "band's line color.",
            "  - Legend:         Includes each band's frequency range "
            "in Hz.",
            "",
            "Right panel — Global change per band:",
            "  - For each band: per-subject difference is computed as "
            "mean(MF, axis=channels)",
            "                   - mean(Rest, axis=channels), then averaged "
            "across subjects.",
            "  - x-axis:         Frequency band (Delta -> Gamma).",
            "  - y-axis:         Mean change in relative power across all "
            "electrodes.",
            "  - Markers:        One per band, colored to match the left "
            "panel.",
            "  - Error bars:     ~95 % CI computed as ±1.96 * SEM across "
            "subjects",
            "                    (normal approximation).",
            "  - Reference:      Dashed horizontal line at y = 0 marks "
            "no change.",
            "  - A thin gray line connects the band-level points to make "
            "the across-band",
            "    pattern easier to follow.",
        ],
    )

    band_fig_lines: List[str] = []
    var_fig_lines: List[str] = []
    for band in FREQUENCY_BANDS:
        bp = os.path.join(out_dir, f"02_band_{band.replace(' ', '_')}.png")
        plot_band_summary(
            group, band, stats_per_band[band], cluster_per_band[band],
            roi_all[band],
            save_path=bp,
            alpha_cluster=alpha,
            n_permutations=n_permutations,
        )
        band_fig_lines.append(f"  {os.path.basename(bp)}")

        vp = os.path.join(out_dir, f"03_variability_{band.replace(' ', '_')}.png")
        plot_subject_variability(group, band, save_path=vp)
        var_fig_lines.append(f"  {os.path.basename(vp)}")

    logger.add(
        "FIGURES 02_band_<BAND>.png — Per-band publication summaries",
        [
            "Generator:          plot_band_summary()",
            "Files generated:",
            *band_fig_lines,
            "",
            "Layout (per band): one figure with two rows.",
            "  Top row (4 topomaps, mne.viz.plot_topomap, cmap='jet'):",
            "    1. Rest — group-mean relative band power.",
            "    2. Mindfulness — group-mean relative band power "
            "(same scale as Rest).",
            "    3. MF - Rest — paired difference; black markers at "
            "electrodes belonging",
            f"       to significant spatial clusters from "
            f"mne.stats.permutation_cluster_1samp_test",
            f"       (n_permutations={n_permutations}, two-sided, "
            f"p < {alpha}).",
            "    4. Cohen's d_z — per-channel paired effect size "
            "(cohens_dz()).",
            "       Black markers at electrodes belonging to significant "
            "spatial clusters",
            "       from mne.stats.permutation_cluster_1samp_test "
            "(same test and parameters",
            "       as panel 3).",
            "    Colorbars under each topomap describe the units.",
            "",
            "  Bottom row (ROI bar plot):",
            "    - Bars: mean relative power per ROI for Rest (blue) and "
            "Mindfulness (red).",
            "    - Error bars: SEM (std with ddof=1 / sqrt(N)).",
            "    - Gray lines: individual subjects (one line per subject "
            "per ROI).",
            "    - Significance stars above each ROI from "
            "scipy.stats.ttest_rel:",
            "        * p < 0.05, ** p < 0.01, *** p < 0.001, n.s. otherwise.",
            "    - Annotated Cohen's d_z per ROI.",
            "    - Legend identifies bars and individual-subject lines.",
        ],
    )

    logger.add(
        "FIGURES 03_variability_<BAND>.png — Per-subject variability",
        [
            "Generator:          plot_subject_variability()",
            "Files generated:",
            *var_fig_lines,
            "",
            "Layout (per band): one figure with two panels.",
            "  Left panel — Paired line plot:",
            "    - One line per subject connecting Rest to Mindfulness "
            "(mean relative power",
            "      averaged across all electrodes).",
            "    - Red line: subjects with MF > Rest. Blue line: subjects "
            "with MF < Rest.",
            "    - Black square line: group mean.",
            "    - Legend distinguishes increasers, decreasers and the "
            "group mean.",
            "  Right panel — Per-subject difference topomaps:",
            "    - Grid of (MF - Rest) topographies, one per subject "
            "(mne.viz.plot_topomap).",
            "    - Diverging colormap 'RdBu_r', symmetric scale clipped at "
            "the 95th percentile",
            "      of |MF - Rest| across all subjects/channels for that band.",
            "    - Each topomap is titled with the subject ID.",
        ],
    )

    log_path = os.path.join(out_dir, "figures_methods.txt")
    logger.write(log_path)

    print(f"  Figures saved in: {os.path.abspath(out_dir)}")
    plt.show()

    return {
        "group": group,
        "stats_per_band": stats_per_band,
        "cluster_per_band": cluster_per_band,
        "roi_results": roi_all,
        "band_contributions": contrib_table,
        "band_contributions_csv": contrib_csv,
        "methods_log": log_path,
    }


# =============================================================================
# Example usage
# =============================================================================
if __name__ == "__main__":
    file_pairs = [
        (f"Sujeto{i}_R_processed.fif", f"Sujeto{i}_MF_processed.fif")
        for i in list(range(1, 12)) + list(range(13, 35)) + [36, 37]
    ]
    results = run_full_analysis(
        file_pairs,
        out_dir="figurespaper",
        n_permutations=5000,
        alpha=0.05,
        relative=True,
    )

STEP 1/4: Loading data and computing PSDs
[S01] Loading Sujeto1_R_processed.fif / Sujeto1_MF_processed.fif
[S02] Loading Sujeto2_R_processed.fif / Sujeto2_MF_processed.fif
[S03] Loading Sujeto3_R_processed.fif / Sujeto3_MF_processed.fif
[S04] Loading Sujeto4_R_processed.fif / Sujeto4_MF_processed.fif
[S05] Loading Sujeto5_R_processed.fif / Sujeto5_MF_processed.fif
[S06] Loading Sujeto6_R_processed.fif / Sujeto6_MF_processed.fif
[S07] Loading Sujeto7_R_processed.fif / Sujeto7_MF_processed.fif
[S08] Loading Sujeto8_R_processed.fif / Sujeto8_MF_processed.fif
[S09] Loading Sujeto9_R_processed.fif / Sujeto9_MF_processed.fif
[S10] Loading Sujeto10_R_processed.fif / Sujeto10_MF_processed.fif
[S11] Loading Sujeto11_R_processed.fif / Sujeto11_MF_processed.fif
[S12] Loading Sujeto13_R_processed.fif / Sujeto13_MF_processed.fif
[S13] Loading Sujeto14_R_processed.fif / Sujeto14_MF_processed.fif
[S14] Loading Sujeto15_R_processed.fif / Sujeto15_MF_processed.fif
[S15] Loading Sujeto16_R_processed.fif

## Connectivity analysis

In [ ]:
"""
Group-level EEG connectivity analysis: Rest vs. Mindfulness
===========================================================

Exhaustive sensor-space connectivity pipeline with:

    Undirected (mne-connectivity, multitaper):
        - Coherence (coh)
        - Imaginary coherence (imcoh)
        - Phase-locking value (plv)
        - Phase-lag index (pli)
        - Weighted phase-lag index (wpli)

    Directed (MVAR-based, statsmodels):
        - Directed Transfer Function (DTF)
        - Partial Directed Coherence (PDC)
        - Granger causality (time-domain, F-test)

    Graph theory (networkx):
        - Node-level: degree, strength, clustering, betweenness,
                      eigenvector centrality, local efficiency
        - Global:     mean strength, mean clustering, char. path length,
                      global efficiency, modularity (Louvain), small-worldness

    Group-level statistics:
        - Edgewise paired permutation test + FDR
        - Network-Based Statistics (NBS, Zalesky 2010) — component-level
          permutation correction
        - Graph-metric paired tests (t + Wilcoxon, Cohen's d_z)

    Plots:
        - Connectivity matrices (R / MF / diff / NBS-thresholded) per band/method
        - Circular/chord diagrams (top-k edges, ROI-grouped)
        - Topographic node-edge plots on the head (using sensor positions)
        - Graph-metric ROI boxplots and band heatmaps
        - Degree/strength topomaps per band

Dependencies
------------
    mne, mne-connectivity, numpy, scipy, matplotlib,
    networkx, statsmodels (for VAR), pandas (optional, summary tables)

Author: improved version
"""

from __future__ import annotations

import os
import warnings
from dataclasses import dataclass, field
from itertools import combinations
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import mne
import networkx as nx
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from mne_connectivity import spectral_connectivity_epochs
from scipy.stats import false_discovery_control, ttest_rel, wilcoxon
from statsmodels.tsa.api import VAR

warnings.filterwarnings("ignore", category=RuntimeWarning)

# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

FREQUENCY_BANDS: Dict[str, Tuple[float, float]] = {
    "Delta":     (0.5,  4),
    "Theta":     (4,    8),
    "Alpha":     (8,   12),
    "Beta":      (12,  18),
    "Fast Beta": (18,  30),
    "Gamma":     (30,  50),
}

ROI_DEFINITIONS: Dict[str, List[str]] = {
    "Frontal":   ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "AF3", "AF4"],
    "Central":   ["FC5", "FC1", "FC2", "FC6", "C3", "Cz", "C4",
                  "CP5", "CP1", "CP2", "CP6"],
    "Parietal":  ["P7", "P3", "Pz", "P4", "P8", "PO3", "PO4"],
    "Occipital": ["O1", "Oz", "O2"],
    "Temporal":  ["T7", "T8", "FT7", "FT8", "TP7", "TP8"],
}

UNDIRECTED_METHODS = ("coh", "imcoh", "plv", "pli", "wpli")
DIRECTED_METHODS   = ("dtf", "pdc", "granger")
ALL_METHODS        = UNDIRECTED_METHODS + DIRECTED_METHODS


# =============================================================================
# Containers
# =============================================================================
@dataclass
class ConnectivityResults:
    """
    Group-level connectivity matrices.

    matrices[method][band] is shape (n_subjects, n_channels, n_channels).
    Undirected methods produce symmetric matrices (diagonal = 0).
    Directed methods produce asymmetric matrices: M[i, j] = i -> j.
    """
    matrices:    Dict[str, Dict[str, np.ndarray]]
    info:        mne.Info
    subject_ids: List[str] = field(default_factory=list)
    conditions:  Tuple[str, str] = ("R", "MF")

    @property
    def n_subjects(self) -> int:
        first_method = next(iter(self.matrices))
        first_band   = next(iter(self.matrices[first_method]))
        return self.matrices[first_method][first_band].shape[0]

    @property
    def n_channels(self) -> int:
        return len(self.info["ch_names"])


# =============================================================================
# Connectivity computation — undirected (mne-connectivity)
# =============================================================================
def _epochs_to_array(epochs: mne.BaseEpochs) -> np.ndarray:
    """Return (n_epochs, n_channels, n_times) array."""
    return epochs.get_data(copy=False)


def compute_undirected_connectivity(
    epochs: mne.BaseEpochs,
    methods: Sequence[str] = UNDIRECTED_METHODS,
    bands: Dict[str, Tuple[float, float]] = FREQUENCY_BANDS,
    mode: str = "multitaper",
    mt_bandwidth: float = 2.0,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Compute undirected spectral connectivity per band.
    Returns: {method: {band: (n_ch, n_ch) symmetric matrix}}.
    """
    sfreq = epochs.info["sfreq"]
    n_ch  = len(epochs.ch_names)

    fmins = [bands[b][0] for b in bands]
    fmaxs = [bands[b][1] for b in bands]
    band_names = list(bands.keys())

    out: Dict[str, Dict[str, np.ndarray]] = {m: {} for m in methods}
    for method in methods:
        con = spectral_connectivity_epochs(
            data=epochs,
            method=method,
            mode=mode,
            sfreq=sfreq,
            fmin=tuple(fmins),
            fmax=tuple(fmaxs),
            faverage=True,
            mt_adaptive=False,
            mt_bandwidth=mt_bandwidth,
            verbose="ERROR",
        )
        # Returned shape: (n_ch * n_ch, n_bands) — lower-triangular flattened
        data = con.get_data(output="dense")  # (n_ch, n_ch, n_bands)
        # mne-connectivity fills only the lower triangle; symmetrize.
        for bi, band in enumerate(band_names):
            mat = data[..., bi]
            mat = mat + mat.T
            np.fill_diagonal(mat, 0.0)
            # imcoh can be negative — keep sign; for the others values are >= 0
            out[method][band] = mat
    return out


# =============================================================================
# Connectivity computation — directed (MVAR / DTF / PDC / Granger)
# =============================================================================
def _band_filter_epochs(epochs: mne.BaseEpochs, fmin: float, fmax: float) -> np.ndarray:
    """Return (n_epochs, n_ch, n_times) band-passed for a given band."""
    ep_filt = epochs.copy().filter(fmin, fmax, verbose="ERROR",
                                   l_trans_bandwidth="auto",
                                   h_trans_bandwidth="auto")
    return ep_filt.get_data(copy=False)


def _fit_mvar(data_concat: np.ndarray, order: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Fit MVAR(p) using statsmodels.VAR.
    data_concat: (n_times, n_channels) — epochs concatenated along time.
    Returns:
        coeffs: (p, n_ch, n_ch) — A_k such that x[t] = sum_k A_k x[t-k] + e
        sigma:  (n_ch, n_ch)   — noise covariance
    """
    model = VAR(data_concat)
    res = model.fit(order, trend="n")
    A = res.coefs                         # (p, n_ch, n_ch)
    sigma = np.cov(res.resid, rowvar=False)
    return A, sigma


def _select_var_order(data_concat: np.ndarray, max_order: int = 12) -> int:
    """Select VAR order by BIC (capped)."""
    try:
        model = VAR(data_concat)
        sel = model.select_order(maxlags=max_order, trend="n")
        order = int(sel.bic) if sel.bic is not None else 6
        return max(1, min(order, max_order))
    except Exception:
        return 6


def _spectral_dtf_pdc(
    A: np.ndarray, sigma: np.ndarray, fs: float,
    band_fmin: float, band_fmax: float, n_freqs: int = 64,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute band-averaged DTF and PDC from MVAR coefficients.

    Returns:
        DTF_band: (n_ch, n_ch)   DTF[i, j] = i -> j  (Kaminski & Blinowska 1991)
        PDC_band: (n_ch, n_ch)   PDC[i, j] = i -> j  (Baccala & Sameshima 2001)
    """
    p, n_ch, _ = A.shape
    freqs = np.linspace(band_fmin, band_fmax, n_freqs)
    z = np.exp(-2j * np.pi * freqs[:, None] / fs)        # (n_freqs, 1)

    # A_bar(f) = I - sum_k A_k * exp(-2*pi*i*f*k/fs)
    A_bar = np.tile(np.eye(n_ch, dtype=complex), (n_freqs, 1, 1))
    for k in range(p):
        # exp coefficient per frequency
        coef = np.exp(-2j * np.pi * freqs * (k + 1) / fs)  # (n_freqs,)
        A_bar -= coef[:, None, None] * A[k][None, :, :]

    # Transfer function H(f) = inv(A_bar)
    H = np.linalg.inv(A_bar)                             # (n_freqs, n_ch, n_ch)

    # DTF: |H_ij(f)|^2 / sum_k |H_ik(f)|^2  (row-normalized — outflow from i)
    H_abs2 = np.abs(H) ** 2
    dtf = H_abs2 / H_abs2.sum(axis=2, keepdims=True)     # i -> j across rows

    # PDC: |A_bar_ij(f)| / sqrt(sum_k |A_bar_kj(f)|^2)  (col-normalized)
    A_abs2 = np.abs(A_bar) ** 2
    pdc = np.abs(A_bar) / np.sqrt(A_abs2.sum(axis=1, keepdims=True))

    # Average across frequencies in band
    return dtf.mean(axis=0).real, pdc.mean(axis=0).real


def _granger_F_matrix(data_concat: np.ndarray, order: int) -> np.ndarray:
    """
    Pairwise time-domain Granger causality F-statistic matrix (vectorized).

    Strategy:
      - Build the lagged design for ALL channels once: shape (n_eff, n_ch * p).
      - For each target j, the restricted model uses lags of channel j only;
        compute its RSS once per j (not per pair).
      - The full model adds lags of source i. We solve all `i != j` full models
        for a given j using a single QR-based update is overkill; instead we
        solve them in a tight loop but with cheap matrix ops on pre-built lags.

    Complexity: O(n_ch^2) solves, but each solve is O(p^2) on tiny matrices
    rather than O((n_eff)*(2p)^2). Roughly 50-200x faster than the naive loop
    on a 64-channel montage.

    Returns (n_ch, n_ch) where M[i, j] = causality from i -> j (F-statistic).
    """
    n_t, n_ch = data_concat.shape
    p = order
    n_eff = n_t - p
    if n_eff <= 2 * p:
        return np.zeros((n_ch, n_ch))

    # Build full lagged design: columns = [ch0_lag1, ..., ch0_lagp, ch1_lag1, ...]
    # Shape: (n_eff, n_ch * p)
    L = np.empty((n_eff, n_ch * p))
    for k in range(p):
        L[:, k * n_ch:(k + 1) * n_ch] = data_concat[p - k - 1:n_t - k - 1, :]
    # Reorder so lags of each channel are contiguous: channel-major
    # (makes slicing per-channel lag blocks faster)
    L = L.reshape(n_eff, p, n_ch).transpose(0, 2, 1).reshape(n_eff, n_ch * p)
    # Now columns 0..p-1 = channel 0 lags, p..2p-1 = channel 1 lags, etc.

    Y = data_concat[p:, :]                           # (n_eff, n_ch) targets

    F = np.zeros((n_ch, n_ch))
    df_num = p
    df_den = n_eff - 2 * p

    for j in range(n_ch):
        y = Y[:, j]
        Lj = L[:, j * p:(j + 1) * p]                 # restricted: own lags

        # Restricted RSS via normal equations (small p × p system)
        try:
            beta_r = np.linalg.solve(Lj.T @ Lj, Lj.T @ y)
            rss_r = np.sum((y - Lj @ beta_r) ** 2)
        except np.linalg.LinAlgError:
            continue

        for i in range(n_ch):
            if i == j:
                continue
            Li = L[:, i * p:(i + 1) * p]
            # Full model: [Lj, Li], 2p columns
            Xf = np.concatenate([Lj, Li], axis=1)
            try:
                XtX = Xf.T @ Xf
                Xty = Xf.T @ y
                beta_f = np.linalg.solve(XtX, Xty)
                rss_f = np.sum((y - Xf @ beta_f) ** 2)
            except np.linalg.LinAlgError:
                continue

            if rss_f > 0 and df_den > 0:
                F[i, j] = ((rss_r - rss_f) / df_num) / (rss_f / df_den)
    return F


def compute_directed_connectivity(
    epochs: mne.BaseEpochs,
    bands: Dict[str, Tuple[float, float]] = FREQUENCY_BANDS,
    var_order: Optional[int] = None,
    max_order: int = 12,
    do_granger: bool = True,
    max_samples: int = 8000,
    verbose: bool = False,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Compute DTF, PDC, and (time-domain) Granger per band.

    Note: time-domain Granger is band-independent in principle, but here
    we fit the MVAR on the band-passed data so the F-statistic reflects
    band-specific dynamics. This is a common, pragmatic choice.

    Speed knobs:
        max_samples: cap on the number of time samples passed to VAR fitting
                     and Granger. With many long epochs this matters a lot.
                     Default 8000 is plenty for a stable VAR with order <= 12.

    Returns: {method: {band: (n_ch, n_ch) asymmetric, M[i,j] = i->j}}
    """
    import time
    fs = epochs.info["sfreq"]
    out: Dict[str, Dict[str, np.ndarray]] = {"dtf": {}, "pdc": {}, "granger": {}}
    if not do_granger:
        out.pop("granger")

    for band, (fmin, fmax) in bands.items():
        t0 = time.time()
        data = _band_filter_epochs(epochs, fmin, fmax)        # (n_ep, n_ch, n_t)
        # Concatenate along time axis: (n_ep * n_t, n_ch)
        n_ep, n_ch, n_t = data.shape
        data_cat = np.concatenate(
            [data[i].T for i in range(n_ep)], axis=0
        )  # (n_ep * n_t, n_ch)

        # Cap length
        if data_cat.shape[0] > max_samples:
            # Take a contiguous middle slice to preserve temporal structure
            start = (data_cat.shape[0] - max_samples) // 2
            data_cat = data_cat[start:start + max_samples]

        order = var_order or _select_var_order(data_cat, max_order=max_order)
        try:
            A, sigma = _fit_mvar(data_cat, order)
            dtf, pdc = _spectral_dtf_pdc(A, sigma, fs, fmin, fmax)
            np.fill_diagonal(dtf, 0.0)
            np.fill_diagonal(pdc, 0.0)
            out["dtf"][band] = dtf
            out["pdc"][band] = pdc
            if do_granger:
                F = _granger_F_matrix(data_cat, order)
                out["granger"][band] = F
        except Exception as e:
            print(f"   ! MVAR failed for {band}: {e}")
            out["dtf"][band] = np.zeros((n_ch, n_ch))
            out["pdc"][band] = np.zeros((n_ch, n_ch))
            if do_granger:
                out["granger"][band] = np.zeros((n_ch, n_ch))
        if verbose:
            print(f"     {band:9s}  order={order}  samples={data_cat.shape[0]}  "
                  f"({time.time() - t0:.1f}s)")
    return out


# =============================================================================
# Loading group connectivity
# =============================================================================
def load_group_connectivity(
    file_pairs: Sequence[Tuple[str, str]],
    methods: Sequence[str] = ALL_METHODS,
    bands: Dict[str, Tuple[float, float]] = FREQUENCY_BANDS,
    var_order: Optional[int] = None,
    verbose: bool = True,
) -> Tuple[ConnectivityResults, ConnectivityResults]:
    """
    Compute connectivity for all subjects, both conditions.
    Returns (results_R, results_MF), each a ConnectivityResults.
    """
    methods_undir = [m for m in methods if m in UNDIRECTED_METHODS]
    methods_dir   = [m for m in methods if m in DIRECTED_METHODS]
    do_granger    = "granger" in methods_dir

    info = None
    subj_ids = []
    matrices_R:  Dict[str, Dict[str, List[np.ndarray]]] = {
        m: {b: [] for b in bands} for m in methods}
    matrices_MF: Dict[str, Dict[str, List[np.ndarray]]] = {
        m: {b: [] for b in bands} for m in methods}

    for i, (f_R, f_MF) in enumerate(file_pairs):
        sid = f"S{i + 1:02d}"
        if verbose:
            print(f"[{sid}] {os.path.basename(f_R)} / {os.path.basename(f_MF)}")
        try:
            ep_R  = mne.read_epochs(f_R,  preload=True, verbose="ERROR")
            ep_MF = mne.read_epochs(f_MF, preload=True, verbose="ERROR")
        except Exception as e:
            print(f"  ! skipped ({e})")
            continue

        # Match channels & order
        common = [c for c in ep_R.ch_names if c in ep_MF.ch_names]
        ep_R  = ep_R.copy().pick(common)
        ep_MF = ep_MF.copy().pick(common)

        # Undirected
        if methods_undir:
            if verbose:
                print("   - undirected (coh/imcoh/plv/pli/wpli) ...")
            u_R  = compute_undirected_connectivity(ep_R,  methods_undir, bands)
            u_MF = compute_undirected_connectivity(ep_MF, methods_undir, bands)
            for m in methods_undir:
                for b in bands:
                    matrices_R [m][b].append(u_R [m][b])
                    matrices_MF[m][b].append(u_MF[m][b])

        # Directed
        if methods_dir:
            if verbose:
                print("   - directed (dtf/pdc/granger) ...")
            d_R  = compute_directed_connectivity(ep_R,  bands,
                                                 var_order=var_order,
                                                 do_granger=do_granger,
                                                 verbose=verbose)
            d_MF = compute_directed_connectivity(ep_MF, bands,
                                                 var_order=var_order,
                                                 do_granger=do_granger,
                                                 verbose=verbose)
            for m in methods_dir:
                for b in bands:
                    matrices_R [m][b].append(d_R [m][b])
                    matrices_MF[m][b].append(d_MF[m][b])

        if info is None:
            info = ep_R.info
        subj_ids.append(sid)

    # Stack lists -> (n_subj, n_ch, n_ch)
    def _stack(d):
        return {m: {b: np.stack(d[m][b]) for b in d[m]} for m in d}

    return (
        ConnectivityResults(_stack(matrices_R),  info, subj_ids),
        ConnectivityResults(_stack(matrices_MF), info, subj_ids),
    )


# =============================================================================
# Statistics: edgewise + NBS
# =============================================================================
def cohens_dz_matrix(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """Paired d_z over subjects, axis=0. X,Y: (n_subj, n_ch, n_ch)."""
    diff = X - Y
    sd = np.std(diff, axis=0, ddof=1)
    sd = np.where(sd == 0, np.nan, sd)
    return np.mean(diff, axis=0) / sd


def edgewise_paired_test(
    X: np.ndarray, Y: np.ndarray,
    alpha: float = 0.05, fdr: bool = True,
) -> Dict[str, np.ndarray]:
    """
    Per-edge paired t-test on (Y - X). Returns t, p, p_fdr, sig, sig_fdr, dz.
    For directed matrices the test is run on each (i,j) including the
    asymmetric pair (j,i) — i.e. on the full off-diagonal.
    """
    n_subj, n_ch, _ = X.shape
    diff = Y - X
    # vectorize ttest_rel via formula (much faster for large n_ch)
    mean = diff.mean(axis=0)
    sd   = diff.std(axis=0, ddof=1)
    se   = sd / np.sqrt(n_subj)
    with np.errstate(divide="ignore", invalid="ignore"):
        t = mean / se
    # two-sided p from t — use scipy via single broadcast
    from scipy.stats import t as student_t
    df = n_subj - 1
    p = 2 * student_t.sf(np.abs(t), df=df)
    p[np.isnan(p)] = 1.0

    # ignore diagonal in FDR
    mask_offdiag = ~np.eye(n_ch, dtype=bool)
    p_flat = p[mask_offdiag]
    p_fdr_flat = false_discovery_control(p_flat, method="bh")
    p_fdr = np.ones_like(p)
    p_fdr[mask_offdiag] = p_fdr_flat

    return {
        "t":       t,
        "p":       p,
        "p_fdr":   p_fdr,
        "sig":     (p     < alpha) & mask_offdiag,
        "sig_fdr": (p_fdr < alpha) & mask_offdiag,
        "dz":      cohens_dz_matrix(Y, X),
    }


def network_based_statistics(
    X: np.ndarray, Y: np.ndarray,
    threshold_p: float = 0.01,
    n_permutations: int = 1000,
    alpha: float = 0.05,
    seed: int = 42,
    tail: str = "both",
) -> Dict[str, np.ndarray]:
    """
    Network-Based Statistics (Zalesky, Fornito, Bullmore, 2010).

    For each permutation of paired sign flips:
      1. Compute t-stat per edge.
      2. Threshold at |t| > t_crit (from threshold_p, two-sided).
      3. Find connected components in the resulting suprathreshold graph.
      4. Record the size of the largest component.

    NBS p-value per observed component = fraction of permutations whose
    largest null component is >= the observed component size.

    Operates on the upper triangle (symmetric assumption). For directed
    matrices, call separately on the (i->j) and (j->i) parts or symmetrize.

    Returns: dict with t_obs, components (list of edge masks), p_components.
    """
    rng = np.random.default_rng(seed)
    n_subj, n_ch, _ = X.shape
    diff = Y - X
    triu = np.triu_indices(n_ch, k=1)

    # Critical t for the chosen threshold
    from scipy.stats import t as student_t
    df = n_subj - 1
    if tail == "both":
        t_crit = student_t.ppf(1 - threshold_p / 2, df)
    else:
        t_crit = student_t.ppf(1 - threshold_p, df)

    def _t_stat(d):
        m = d.mean(axis=0)
        s = d.std(axis=0, ddof=1)
        return m / (s / np.sqrt(n_subj))

    t_obs_full = _t_stat(diff)
    np.nan_to_num(t_obs_full, copy=False)

    # Build observed suprathreshold graph (upper tri)
    def _components(t_mat: np.ndarray) -> List[np.ndarray]:
        if tail == "both":
            supra = np.abs(t_mat) > t_crit
        elif tail == "pos":
            supra = t_mat > t_crit
        else:
            supra = t_mat < -t_crit
        # Symmetrize for undirected component finding
        supra_u = supra.copy()
        supra_u[np.tril_indices(n_ch)] = False
        G = nx.Graph()
        G.add_nodes_from(range(n_ch))
        ii, jj = np.where(supra_u)
        G.add_edges_from(zip(ii, jj))
        comps = []
        for cc in nx.connected_components(G):
            sub = G.subgraph(cc)
            if sub.number_of_edges() == 0:
                continue
            mask = np.zeros((n_ch, n_ch), dtype=bool)
            for u, v in sub.edges():
                mask[u, v] = True
                mask[v, u] = True
            comps.append(mask)
        return comps

    obs_comps = _components(t_obs_full)
    obs_sizes = sorted([c[np.triu_indices(n_ch, 1)].sum() for c in obs_comps],
                       reverse=True)

    # Permutations: sign-flip each subject's diff
    null_max = np.zeros(n_permutations)
    for k in range(n_permutations):
        signs = rng.choice([-1, 1], size=n_subj)
        d_perm = diff * signs[:, None, None]
        t_perm = _t_stat(d_perm)
        np.nan_to_num(t_perm, copy=False)
        comps = _components(t_perm)
        if comps:
            null_max[k] = max(c[np.triu_indices(n_ch, 1)].sum() for c in comps)
        else:
            null_max[k] = 0

    # p per observed component
    p_components = []
    for c in obs_comps:
        size = c[np.triu_indices(n_ch, 1)].sum()
        p = (np.sum(null_max >= size) + 1) / (n_permutations + 1)
        p_components.append(p)

    sig_mask = np.zeros((n_ch, n_ch), dtype=bool)
    for c, p in zip(obs_comps, p_components):
        if p < alpha:
            sig_mask |= c

    return {
        "t_obs":          t_obs_full,
        "t_crit":         float(t_crit),
        "components":     obs_comps,
        "component_sizes": obs_sizes,
        "p_components":   p_components,
        "sig_mask":       sig_mask,
        "null_max":       null_max,
    }


# =============================================================================
# Graph theory
# =============================================================================
def _abs_symmetric(W: np.ndarray) -> np.ndarray:
    """Return symmetric, non-negative version of W with zero diagonal."""
    M = (np.abs(W) + np.abs(W.T)) / 2.0
    np.fill_diagonal(M, 0.0)
    return M


def graph_metrics(
    W: np.ndarray,
    proportional_threshold: float = 0.2,
) -> Dict[str, np.ndarray]:
    """
    Compute graph metrics on a (sparsified, weighted, undirected) matrix.

    Steps:
        1. Symmetrize and abs-value (for compatibility with all methods).
        2. Apply proportional thresholding: keep top X% strongest edges.
        3. Compute weighted graph metrics.

    Returns dict with node-level (n_ch,) and global (scalar) metrics.
    """
    M = _abs_symmetric(W)

    # Proportional threshold (top-X% of edges)
    n = M.shape[0]
    triu = np.triu_indices(n, 1)
    weights = M[triu]
    if weights.size == 0:
        return {"strength": np.zeros(n), "global": {}}
    k = max(1, int(np.ceil(proportional_threshold * weights.size)))
    cutoff = np.sort(weights)[-k]
    M_thr = np.where(M >= cutoff, M, 0.0)
    np.fill_diagonal(M_thr, 0.0)

    G = nx.from_numpy_array(M_thr)

    # Node-level
    strength = np.array(M_thr.sum(axis=1))
    degree   = np.array([d for _, d in G.degree()])
    clustering_dict = nx.clustering(G, weight="weight")
    clustering = np.array([clustering_dict[i] for i in range(n)])

    try:
        betw_dict = nx.betweenness_centrality(G, weight="weight", normalized=True)
    except Exception:
        betw_dict = {i: 0 for i in range(n)}
    betweenness = np.array([betw_dict[i] for i in range(n)])

    try:
        eig_dict = nx.eigenvector_centrality_numpy(G, weight="weight")
        eigenvec = np.array([eig_dict[i] for i in range(n)])
    except Exception:
        eigenvec = np.zeros(n)

    # Local efficiency (weighted): mean of harmonic mean of weights in neighborhood
    try:
        # networkx doesn't ship a vectorized weighted local efficiency, do via subgraphs
        local_eff = np.zeros(n)
        for v in G.nodes():
            neigh = list(G.neighbors(v))
            if len(neigh) < 2:
                continue
            sub = G.subgraph(neigh)
            try:
                local_eff[v] = nx.global_efficiency(sub)
            except Exception:
                local_eff[v] = 0.0
    except Exception:
        local_eff = np.zeros(n)

    # Global metrics
    global_metrics = {
        "mean_strength":   float(strength.mean()),
        "mean_clustering": float(clustering.mean()),
        "global_efficiency": float(nx.global_efficiency(G)),
        "transitivity":    float(nx.transitivity(G)),
    }

    # Characteristic path length on largest connected component
    if G.number_of_edges() > 0:
        components = list(nx.connected_components(G))
        largest = max(components, key=len)
        Gc = G.subgraph(largest)
        try:
            global_metrics["char_path_length"] = float(
                nx.average_shortest_path_length(Gc, weight=None)
            )
        except Exception:
            global_metrics["char_path_length"] = np.nan
    else:
        global_metrics["char_path_length"] = np.nan

    # Modularity (greedy approximation; Louvain via networkx community)
    try:
        from networkx.algorithms.community import greedy_modularity_communities, modularity
        comms = list(greedy_modularity_communities(G, weight="weight"))
        global_metrics["modularity"]   = float(modularity(G, comms, weight="weight"))
        global_metrics["n_communities"] = float(len(comms))
    except Exception:
        global_metrics["modularity"]   = np.nan
        global_metrics["n_communities"] = np.nan

    # Small-worldness sigma: skip (computationally heavy & unstable for small n_ch).
    return {
        "strength":     strength,
        "degree":       degree,
        "clustering":   clustering,
        "betweenness":  betweenness,
        "eigenvector":  eigenvec,
        "local_eff":    local_eff,
        "global":       global_metrics,
    }


def graph_metrics_group(
    matrices: np.ndarray,
    proportional_threshold: float = 0.2,
) -> Dict[str, np.ndarray]:
    """Apply graph_metrics across subjects. matrices: (n_subj, n_ch, n_ch)."""
    n_subj = matrices.shape[0]
    out_node = {k: [] for k in
                ("strength", "degree", "clustering",
                 "betweenness", "eigenvector", "local_eff")}
    out_global_keys = None
    out_global = []
    for s in range(n_subj):
        gm = graph_metrics(matrices[s], proportional_threshold)
        for k in out_node:
            out_node[k].append(gm[k])
        if out_global_keys is None:
            out_global_keys = list(gm["global"].keys())
        out_global.append([gm["global"][k] for k in out_global_keys])
    out_node = {k: np.stack(v) for k, v in out_node.items()}
    out_global = np.array(out_global)  # (n_subj, n_global)
    return {"node": out_node, "global_keys": out_global_keys, "global": out_global}


def compare_graph_metrics(
    R_metrics: Dict, MF_metrics: Dict, alpha: float = 0.05,
) -> Dict[str, Dict[str, float]]:
    """Paired tests on global graph metrics."""
    out = {}
    for i, k in enumerate(R_metrics["global_keys"]):
        a = R_metrics["global"][:, i]
        b = MF_metrics["global"][:, i]
        a_clean = a[~(np.isnan(a) | np.isnan(b))]
        b_clean = b[~(np.isnan(a) | np.isnan(b))]
        if len(a_clean) < 3:
            out[k] = dict(t=np.nan, p=np.nan, dz=np.nan,
                          mean_R=np.nanmean(a), mean_MF=np.nanmean(b))
            continue
        t, p = ttest_rel(b_clean, a_clean)
        try:
            _, p_w = wilcoxon(b_clean, a_clean)
        except ValueError:
            p_w = np.nan
        dz = (b_clean - a_clean).mean() / (b_clean - a_clean).std(ddof=1)
        out[k] = dict(
            t=float(t), p=float(p), p_wilcoxon=float(p_w), dz=float(dz),
            mean_R=float(a_clean.mean()), mean_MF=float(b_clean.mean()),
            sig=p < alpha,
        )
    return out


# =============================================================================
# Plotting
# =============================================================================
def _channel_xy(info: mne.Info) -> np.ndarray:
    """Return (n_ch, 2) topomap coordinates for plotting."""
    return np.array([
        mne.channels.layout._find_topomap_coords(info, picks=[ch])[0]
        for ch in range(len(info["ch_names"]))
    ])


def plot_connectivity_matrices(
    R_matrices: Dict[str, np.ndarray],
    MF_matrices: Dict[str, np.ndarray],
    nbs_results: Dict[str, Dict],
    info: mne.Info,
    method: str,
    save_path: Optional[str] = None,
):
    """One figure per method: rows = bands, cols = R / MF / diff / NBS-sig."""
    bands = list(FREQUENCY_BANDS.keys())
    n_rows, n_cols = len(bands), 4
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(3.4 * n_cols, 2.7 * n_rows),
                             constrained_layout=True)
    fig.suptitle(f"Connectivity matrices — {method.upper()}",
                 fontsize=14, fontweight="bold")

    ch_names = info["ch_names"]
    n_ch = len(ch_names)
    ticks = np.arange(0, n_ch, max(1, n_ch // 12))

    for r, band in enumerate(bands):
        R_mean  = R_matrices[band].mean(axis=0)
        MF_mean = MF_matrices[band].mean(axis=0)
        diff    = MF_mean - R_mean
        nbs     = nbs_results[band]["sig_mask"]

        vmax = max(R_mean.max(), MF_mean.max())
        vdmax = max(abs(diff.min()), abs(diff.max())) or 1e-12

        im0 = axes[r, 0].imshow(R_mean,  cmap="viridis", vmin=0,    vmax=vmax)
        im1 = axes[r, 1].imshow(MF_mean, cmap="viridis", vmin=0,    vmax=vmax)
        im2 = axes[r, 2].imshow(diff, cmap="RdBu_r", vmin=-vdmax, vmax=vdmax)
        # NBS panel: diff masked to significant component
        masked_diff = np.where(nbs, diff, 0.0)
        im3 = axes[r, 3].imshow(masked_diff, cmap="RdBu_r", vmin=-vdmax, vmax=vdmax)

        if r == 0:
            axes[r, 0].set_title("Rest")
            axes[r, 1].set_title("Mindfulness")
            axes[r, 2].set_title("MF − Rest")
            axes[r, 3].set_title("NBS-significant")

        axes[r, 0].set_ylabel(f"{band}\n({FREQUENCY_BANDS[band][0]}–"
                              f"{FREQUENCY_BANDS[band][1]} Hz)",
                              fontweight="bold")
        for ax, im in zip(axes[r], (im0, im1, im2, im3)):
            ax.set_xticks(ticks); ax.set_yticks(ticks)
            ax.set_xticklabels([ch_names[i] for i in ticks],
                               rotation=90, fontsize=6)
            ax.set_yticklabels([ch_names[i] for i in ticks], fontsize=6)
        fig.colorbar(im2, ax=axes[r, 2], shrink=0.7, pad=0.02)

    if save_path:
        fig.savefig(save_path)
    return fig


def plot_circular_diagram(
    matrix: np.ndarray,
    info: mne.Info,
    title: str,
    top_k: int = 50,
    rois: Dict[str, List[str]] = ROI_DEFINITIONS,
    save_path: Optional[str] = None,
    cmap: str = "RdBu_r",
    diverging: bool = True,
):
    """Circular/chord diagram of top-k strongest edges, ROI-grouped."""
    ch_names = info["ch_names"]
    n_ch = len(ch_names)

    # Order channels by ROI
    order = []
    roi_labels = []
    for roi, chans in rois.items():
        for c in chans:
            if c in ch_names:
                order.append(ch_names.index(c))
                roi_labels.append(roi)
    # Append remaining channels
    leftover = [i for i in range(n_ch) if i not in order]
    order.extend(leftover)
    roi_labels.extend(["Other"] * len(leftover))

    # Place nodes on circle
    angles = np.linspace(0, 2 * np.pi, len(order), endpoint=False)
    coords = np.column_stack([np.cos(angles), np.sin(angles)])

    # Pick top-k edges from upper triangle (by absolute value)
    M = matrix[np.ix_(order, order)]
    triu = np.triu_indices(len(order), 1)
    vals = M[triu]
    if top_k > vals.size:
        top_k = vals.size
    top_idx = np.argsort(np.abs(vals))[::-1][:top_k]
    edge_i = triu[0][top_idx]
    edge_j = triu[1][top_idx]
    edge_v = vals[top_idx]

    fig, ax = plt.subplots(figsize=(8.5, 8.5))
    if diverging:
        vmax = np.max(np.abs(edge_v)) or 1e-12
        norm = Normalize(vmin=-vmax, vmax=vmax)
    else:
        norm = Normalize(vmin=edge_v.min(), vmax=edge_v.max())
    cmap_obj = mpl.colormaps.get_cmap(cmap)

    # Draw edges as quadratic Bézier toward center
    for i, j, v in zip(edge_i, edge_j, edge_v):
        x0, y0 = coords[i]; x1, y1 = coords[j]
        # Bezier control point: pull toward center
        ctrl = np.array([0, 0])
        t = np.linspace(0, 1, 32)
        bx = (1 - t)**2 * x0 + 2 * (1 - t) * t * ctrl[0] + t**2 * x1
        by = (1 - t)**2 * y0 + 2 * (1 - t) * t * ctrl[1] + t**2 * y1
        ax.plot(bx, by, color=cmap_obj(norm(v)),
                lw=0.7 + 2.5 * (abs(v) / (np.abs(edge_v).max() + 1e-12)),
                alpha=0.75, zorder=2)

    # ROI-colored nodes
    roi_colors = {
        "Frontal": "#E45756", "Central": "#F58518", "Parietal": "#54A24B",
        "Occipital": "#4C78A8", "Temporal": "#B279A2", "Other": "#888888",
    }
    for k, (xy, roi) in enumerate(zip(coords, roi_labels)):
        ax.scatter(*xy, s=80, c=roi_colors.get(roi, "gray"),
                   edgecolors="white", zorder=4, linewidths=1.0)
        # Label outside circle
        lbl_xy = xy * 1.12
        ax.text(*lbl_xy, ch_names[order[k]],
                ha="center", va="center", fontsize=7,
                rotation=np.degrees(angles[k]) - 90 if angles[k] > np.pi else
                        np.degrees(angles[k]) + 90,
                rotation_mode="anchor")

    # ROI legend
    handles = [plt.Line2D([0], [0], marker="o", color="w",
                          markerfacecolor=c, markersize=10, label=r)
               for r, c in roi_colors.items() if r in roi_labels]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1.0),
              frameon=False)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.3])
    fig.colorbar(sm, cax=cax, label="Edge value")

    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold")

    if save_path:
        fig.savefig(save_path)
    return fig


def plot_node_edge_topo(
    matrix: np.ndarray,
    info: mne.Info,
    title: str,
    edge_threshold: float = 0.0,
    top_k: Optional[int] = 100,
    node_metric: Optional[np.ndarray] = None,
    save_path: Optional[str] = None,
    cmap_edges: str = "RdBu_r",
    cmap_nodes: str = "viridis",
    diverging: bool = True,
):
    """
    Topographic node-edge plot: nodes at sensor positions, edges as lines
    with color/width proportional to connectivity value.
    """
    pos = _channel_xy(info)                          # (n_ch, 2)
    n_ch = pos.shape[0]

    fig, ax = plt.subplots(figsize=(8, 7.5))
    # Draw head outline using a fake topomap
    fake = np.zeros(n_ch)
    mne.viz.plot_topomap(fake, info, axes=ax, show=False,
                         cmap="Greys", contours=0, sensors=False,
                         outlines="head")
    # plot_topomap draws — wipe colors, keep outline
    for coll in ax.collections:
        coll.remove()
    # Re-add outline
    mne.viz.plot_topomap(fake, info, axes=ax, show=False,
                         cmap="Greys", contours=0, sensors=False,
                         outlines="head")

    # Edges
    triu = np.triu_indices(n_ch, 1)
    vals = matrix[triu]
    if top_k is not None and top_k < vals.size:
        top_idx = np.argsort(np.abs(vals))[::-1][:top_k]
    else:
        top_idx = np.where(np.abs(vals) > edge_threshold)[0]
    e_i = triu[0][top_idx]; e_j = triu[1][top_idx]; e_v = vals[top_idx]

    if diverging and len(e_v):
        vmax = np.max(np.abs(e_v)) or 1e-12
        norm = Normalize(vmin=-vmax, vmax=vmax)
    elif len(e_v):
        norm = Normalize(vmin=e_v.min(), vmax=e_v.max())
    else:
        norm = Normalize(vmin=0, vmax=1)

    cmap_e = mpl.colormaps.get_cmap(cmap_edges)
    segments = [[(pos[i, 0], pos[i, 1]), (pos[j, 0], pos[j, 1])]
                for i, j in zip(e_i, e_j)]
    if len(segments):
        widths = 0.5 + 3.0 * (np.abs(e_v) / (np.abs(e_v).max() + 1e-12))
        lc = LineCollection(segments, colors=[cmap_e(norm(v)) for v in e_v],
                            linewidths=widths, alpha=0.7, zorder=2)
        ax.add_collection(lc)

    # Nodes
    if node_metric is not None and node_metric.max() > node_metric.min():
        sizes = 60 + 250 * (node_metric - node_metric.min()) / \
                (node_metric.max() - node_metric.min())
        sc = ax.scatter(pos[:, 0], pos[:, 1], s=sizes, c=node_metric,
                        cmap=cmap_nodes, edgecolors="black", linewidths=0.6,
                        zorder=4)
        fig.colorbar(sc, ax=ax, shrink=0.6, label="Node strength")
    else:
        ax.scatter(pos[:, 0], pos[:, 1], s=80, c="lightgray",
                   edgecolors="black", linewidths=0.6, zorder=4)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_aspect("equal"); ax.axis("off")

    if len(e_v):
        sm = plt.cm.ScalarMappable(cmap=cmap_e, norm=norm)
        fig.colorbar(sm, ax=ax, shrink=0.6, label="Edge value", location="left")

    if save_path:
        fig.savefig(save_path)
    return fig


def plot_graph_metric_heatmap(
    metric_results: Dict[str, Dict[str, Dict[str, float]]],
    save_path: Optional[str] = None,
):
    """Heatmap: rows = bands, cols = global graph metrics, cell = Cohen's d_z."""
    bands = list(metric_results.keys())
    keys = list(next(iter(metric_results.values())).keys())

    dz_mat = np.array([[metric_results[b][k].get("dz", np.nan) for k in keys]
                       for b in bands])
    p_mat  = np.array([[metric_results[b][k].get("p",  np.nan) for k in keys]
                       for b in bands])

    fig, ax = plt.subplots(figsize=(1.6 * len(keys), 0.7 * len(bands) + 1.5))
    vmax = np.nanmax(np.abs(dz_mat)) or 1e-12
    im = ax.imshow(dz_mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=30, ha="right")
    ax.set_yticks(range(len(bands)))
    ax.set_yticklabels(bands)
    ax.set_title("Global graph metrics: effect sizes (Cohen's $d_z$, MF − Rest)",
                 fontweight="bold")

    for i in range(len(bands)):
        for j in range(len(keys)):
            p = p_mat[i, j]
            if np.isnan(p):
                star = ""
            elif p < 0.001: star = "***"
            elif p < 0.01:  star = "**"
            elif p < 0.05:  star = "*"
            else:           star = ""
            val = dz_mat[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:+.2f}\n{star}",
                        ha="center", va="center", fontsize=8,
                        color="white" if abs(val) > vmax * 0.6 else "black")
    fig.colorbar(im, ax=ax, label="Cohen's $d_z$", shrink=0.8)
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_node_metric_topomaps(
    R_node: Dict[str, np.ndarray],
    MF_node: Dict[str, np.ndarray],
    info: mne.Info,
    band: str,
    method: str,
    save_path: Optional[str] = None,
):
    """Topomaps of node-level graph metrics, R / MF / diff."""
    metrics = ["strength", "clustering", "betweenness", "eigenvector"]
    n_rows, n_cols = 3, len(metrics)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.6 * n_rows),
                             constrained_layout=True)
    fig.suptitle(f"Node-level graph metrics — {method.upper()} / {band}",
                 fontsize=13, fontweight="bold")

    for c, m in enumerate(metrics):
        R_avg  = R_node[m].mean(axis=0)
        MF_avg = MF_node[m].mean(axis=0)
        diff = MF_avg - R_avg

        vmin = min(R_avg.min(), MF_avg.min())
        vmax = max(R_avg.max(), MF_avg.max())
        vdmax = max(abs(diff.min()), abs(diff.max())) or 1e-12

        for r, (data, cmap, vlim, label) in enumerate([
            (R_avg,  "viridis", (vmin, vmax),    "Rest"),
            (MF_avg, "viridis", (vmin, vmax),    "Mindfulness"),
            (diff,   "RdBu_r",  (-vdmax, vdmax), "MF − Rest"),
        ]):
            mne.viz.plot_topomap(data, info, axes=axes[r, c],
                                 cmap=cmap, vlim=vlim, show=False,
                                 contours=0, sensors=True, outlines="head")
            if r == 0:
                axes[r, c].set_title(m, fontsize=11, fontweight="bold", pad=4)
            if c == 0:
                axes[r, c].set_ylabel(label, fontsize=11, fontweight="bold")

    if save_path:
        fig.savefig(save_path)
    return fig


# =============================================================================
# Top-level driver
# =============================================================================
def run_full_connectivity_analysis(
    file_pairs: Sequence[Tuple[str, str]],
    out_dir: str = "connectivity_figures",
    methods: Sequence[str] = ALL_METHODS,
    n_permutations_nbs: int = 1000,
    nbs_threshold_p: float = 0.01,
    alpha: float = 0.05,
    proportional_threshold: float = 0.2,
    var_order: Optional[int] = None,
):
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 72)
    print("STEP 1/5: Computing connectivity (this is the slow part)")
    print("=" * 72)
    R, MF = load_group_connectivity(
        file_pairs, methods=methods, var_order=var_order)
    print(f"  -> {R.n_subjects} subjects, {R.n_channels} channels")

    print("\n" + "=" * 72)
    print("STEP 2/5: Edgewise stats + NBS")
    print("=" * 72)
    edge_stats: Dict[str, Dict[str, Dict[str, np.ndarray]]] = {}
    nbs_stats:  Dict[str, Dict[str, Dict[str, np.ndarray]]] = {}
    for m in methods:
        edge_stats[m] = {}
        nbs_stats[m]  = {}
        for b in FREQUENCY_BANDS:
            X, Y = R.matrices[m][b], MF.matrices[m][b]
            edge_stats[m][b] = edgewise_paired_test(X, Y, alpha=alpha)
            nbs_stats[m][b]  = network_based_statistics(
                X, Y,
                threshold_p=nbs_threshold_p,
                n_permutations=n_permutations_nbs,
                alpha=alpha,
            )
            n_sig_uncorr = edge_stats[m][b]["sig"].sum() // 2
            n_sig_fdr    = edge_stats[m][b]["sig_fdr"].sum() // 2
            n_nbs        = nbs_stats[m][b]["sig_mask"].sum() // 2
            print(f"  {m:8s} {b:9s}  edges: uncorr={n_sig_uncorr:4d}  "
                  f"FDR={n_sig_fdr:4d}  NBS={n_nbs:4d}")

    print("\n" + "=" * 72)
    print("STEP 3/5: Graph theory metrics")
    print("=" * 72)
    # Use coh as the "primary" undirected matrix for graph metrics
    primary = "coh" if "coh" in methods else methods[0]
    graph_R: Dict[str, Dict] = {}
    graph_MF: Dict[str, Dict] = {}
    graph_compare: Dict[str, Dict[str, Dict[str, float]]] = {}
    for b in FREQUENCY_BANDS:
        gR  = graph_metrics_group(R.matrices[primary][b],
                                  proportional_threshold=proportional_threshold)
        gMF = graph_metrics_group(MF.matrices[primary][b],
                                  proportional_threshold=proportional_threshold)
        graph_R [b] = gR
        graph_MF[b] = gMF
        cmp = compare_graph_metrics(gR, gMF, alpha=alpha)
        graph_compare[b] = cmp
        sig_keys = [k for k, v in cmp.items() if v.get("sig", False)]
        print(f"  {b:9s}  significant globals: {sig_keys}")

    print("\n" + "=" * 72)
    print("STEP 4/5: Generating figures")
    print("=" * 72)

    # 1) Connectivity matrices per method
    for m in methods:
        plot_connectivity_matrices(
            R.matrices[m], MF.matrices[m], nbs_stats[m], R.info, m,
            save_path=os.path.join(out_dir, f"01_matrices_{m}.png"),
        )
        plt.close("all")

    # 2) Circular + topo node-edge per band, primary method
    for b in FREQUENCY_BANDS:
        diff = MF.matrices[primary][b].mean(0) - R.matrices[primary][b].mean(0)
        plot_circular_diagram(
            diff, R.info,
            title=f"Top-50 edge differences (MF − Rest)\n{primary.upper()} / {b}",
            top_k=50, diverging=True,
            save_path=os.path.join(out_dir, f"02_circular_{primary}_{b}.png"),
        )
        plot_node_edge_topo(
            diff, R.info,
            title=f"Edge differences on head\n{primary.upper()} / {b}",
            top_k=80,
            node_metric=graph_MF[b]["node"]["strength"].mean(0)
                        - graph_R[b]["node"]["strength"].mean(0),
            save_path=os.path.join(out_dir, f"03_topo_{primary}_{b}.png"),
        )
        # Node-metric topomaps
        plot_node_metric_topomaps(
            graph_R[b]["node"], graph_MF[b]["node"], R.info, b, primary,
            save_path=os.path.join(out_dir, f"04_node_metrics_{primary}_{b}.png"),
        )
        plt.close("all")

    # 3) Global graph metrics heatmap
    plot_graph_metric_heatmap(
        graph_compare,
        save_path=os.path.join(out_dir, "05_global_metrics_heatmap.png"),
    )
    plt.close("all")

    print(f"  Figures saved in: {os.path.abspath(out_dir)}")

    print("\n" + "=" * 72)
    print("STEP 5/5: Done.")
    print("=" * 72)

    return {
        "R": R, "MF": MF,
        "edge_stats": edge_stats,
        "nbs_stats":  nbs_stats,
        "graph_R": graph_R, "graph_MF": graph_MF,
        "graph_compare": graph_compare,
    }


# =============================================================================
# Example usage
# =============================================================================
if __name__ == "__main__":
    file_pairs = [
        (f"Sujeto{i}_R_processed.fif", f"Sujeto{i}_MF_processed.fif")
        for i in list(range(1, 12)) + list(range(13, 35)) + [36, 37]
        # for i in list(range(1, 2)) 
        ]
    results = run_full_connectivity_analysis(
        file_pairs,
        out_dir="connectivity_figures",
        methods=ALL_METHODS,
        n_permutations_nbs=1000,
        nbs_threshold_p=0.01,
        alpha=0.05,
        proportional_threshold=0.2,
    )

STEP 1/5: Computing connectivity (this is the slow part)
[S01] Sujeto1_R_processed.fif / Sujeto1_MF_processed.fif
   - undirected (coh/imcoh/plv/pli/wpli) ...
   - directed (dtf/pdc/granger) ...
     Delta      order=6  samples=8000  (14.8s)
     Theta      order=3  samples=8000  (14.4s)
     Alpha      order=6  samples=8000  (11.6s)
     Beta       order=6  samples=8000  (8.7s)
     Fast Beta  order=6  samples=8000  (8.6s)
     Gamma      order=6  samples=8000  (9.6s)
     Delta      order=6  samples=8000  (22.4s)
     Theta      order=6  samples=8000  (17.0s)
     Alpha      order=6  samples=8000  (12.1s)
     Beta       order=6  samples=8000  (13.9s)
     Fast Beta  order=6  samples=8000  (11.1s)
     Gamma      order=6  samples=8000  (12.8s)
[S02] Sujeto2_R_processed.fif / Sujeto2_MF_processed.fif
   - undirected (coh/imcoh/plv/pli/wpli) ...
   - directed (dtf/pdc/granger) ...
     Delta      order=4  samples=8000  (12.9s)
     Theta      order=6  samples=8000  (7.6s)
     Alpha   

## Other analysis

In [1]:
"""
Advanced EEG analysis: Rest vs. Mindfulness
============================================

Beyond power and connectivity. Six methods, group-level, paired stats.

    1. COMPLEXITY (signal-domain, nonlinear)
       - Sample entropy, Permutation entropy, Multiscale entropy (curve)
       - Spectral entropy, Approximate entropy
       - Higuchi fractal dimension
       - Detrended Fluctuation Analysis (DFA exponent)
       - Lempel-Ziv complexity (binary, median-thresholded)

    2. MICROSTATES (spatiotemporal)
       - GFP-peak topography extraction
       - Modified k-means (Pascual-Marqui) clustering, k=4 canonical
       - Per-subject backfitting -> coverage, occurrence, mean duration
       - Transition probability matrices

    3. APERIODIC vs PERIODIC (FOOOF / specparam)
       - Aperiodic offset and exponent per channel
       - Periodic peak parameters (CF, PW, BW) per band
       - Goodness-of-fit (R^2, error)

    4. CROSS-FREQUENCY COUPLING
       - Modulation Index (Tort 2010): theta-gamma, alpha-gamma, theta-beta
       - Per-channel + group mean topographies

    5. ASYMMETRY / LATERALIZATION
       - Frontal alpha asymmetry (FAA): classic ln(R) - ln(L)
       - All-band asymmetry indices for symmetric channel pairs

    6. INDIVIDUAL ALPHA FREQUENCY (IAF)
       - Peak alpha frequency (PAF) and center of gravity (CoG)
       - Per-subject, per-region; used to define personalized alpha bands

    7. TIME-RESOLVED SPECTRAL DYNAMICS
       - Sliding-window PSD -> band-power time series
       - Power variability (SD over time), drift slopes

Skipped (with reason):
    - ERSP/ITC: event-related, requires stimulus-locked epochs.
    - Lyapunov: noise-sensitive, unstable on real EEG epochs.
    - Source localization: requires forward model + BEM + montage
      co-registration; a separate pipeline. A documented stub is included.

Dependencies
------------
    mne, numpy, scipy, matplotlib
    fooof (or specparam)
    antropy        (entropy + fractal measures)
    scikit-learn   (k-means)
"""

from __future__ import annotations

import os
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import antropy as ant
import matplotlib as mpl
import matplotlib.pyplot as plt
import mne
import numpy as np
from scipy.signal import hilbert, welch
from scipy.stats import false_discovery_control, ttest_rel, wilcoxon
from sklearn.cluster import KMeans

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    from fooof import FOOOFGroup

warnings.filterwarnings("ignore", category=RuntimeWarning)

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

FREQUENCY_BANDS: Dict[str, Tuple[float, float]] = {
    "Delta":     (0.5,  4),
    "Theta":     (4,    8),
    "Alpha":     (8,   12),
    "Beta":      (12,  18),
    "Fast Beta": (18,  30),
    "Gamma":     (30,  50),
}

ROI_DEFINITIONS: Dict[str, List[str]] = {
    "Frontal":   ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "AF3", "AF4"],
    "Central":   ["FC5", "FC1", "FC2", "FC6", "C3", "Cz", "C4",
                  "CP5", "CP1", "CP2", "CP6"],
    "Parietal":  ["P7", "P3", "Pz", "P4", "P8", "PO3", "PO4"],
    "Occipital": ["O1", "Oz", "O2"],
    "Temporal":  ["T7", "T8", "FT7", "FT8", "TP7", "TP8"],
}

# Frontal alpha asymmetry standard pairs (left, right)
FAA_PAIRS = [("F3", "F4"), ("F7", "F8"), ("Fp1", "Fp2"), ("AF3", "AF4")]


# =============================================================================
# Helpers
# =============================================================================
def cohens_dz(x: np.ndarray, y: np.ndarray, axis: int = 0) -> np.ndarray:
    diff = x - y
    sd = np.std(diff, axis=axis, ddof=1)
    sd = np.where(sd == 0, np.nan, sd)
    return np.mean(diff, axis=axis) / sd


def paired_test_summary(R: np.ndarray, MF: np.ndarray) -> Dict[str, float]:
    """Return paired t, Wilcoxon, dz on subject-level paired vectors."""
    a = np.asarray(R); b = np.asarray(MF)
    valid = ~(np.isnan(a) | np.isnan(b))
    a, b = a[valid], b[valid]
    if len(a) < 3:
        return dict(t=np.nan, p=np.nan, p_w=np.nan, dz=np.nan,
                    mean_R=np.nanmean(R), mean_MF=np.nanmean(MF))
    t, p = ttest_rel(b, a)
    try:
        _, p_w = wilcoxon(b, a)
    except ValueError:
        p_w = np.nan
    return dict(
        t=float(t), p=float(p), p_w=float(p_w) if not np.isnan(p_w) else np.nan,
        dz=float(cohens_dz(b, a)),
        mean_R=float(a.mean()), mean_MF=float(b.mean()),
    )


def _add_stars(p: float) -> str:
    if np.isnan(p): return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""


# =============================================================================
# 1. COMPLEXITY MEASURES
# =============================================================================
def compute_complexity_per_channel(
    data: np.ndarray, sfreq: float,
    measures: Sequence[str] = ("samp_en", "perm_en", "spec_en",
                                "higuchi", "dfa", "lziv"),
    max_samples_slow: int = 5000,
    max_samples_lziv: int = 10000,
) -> Dict[str, np.ndarray]:
    """
    Compute complexity measures per channel on a (n_channels, n_times) signal.

    Returns dict {measure: (n_channels,)}.

    Speed notes:
        - Sample entropy is O(n^2); we cap at `max_samples_slow` (~5000).
        - Lempel-Ziv on binarized data also scales poorly; we cap at
          `max_samples_lziv` (~10000) by default. Lower for faster runs.
        - The other measures (perm/spectral entropy, Higuchi FD, DFA) are
          fast enough to run on the full series.
    """
    n_ch, n_t = data.shape
    cap_slow = min(n_t, max_samples_slow)
    cap_lziv = min(n_t, max_samples_lziv)

    out = {m: np.zeros(n_ch) for m in measures}
    for ch in range(n_ch):
        x_full = data[ch]
        x_slow = x_full[:cap_slow]
        x_lziv = x_full[:cap_lziv]
        try:
            if "samp_en" in measures:
                out["samp_en"][ch] = ant.sample_entropy(x_slow, order=2)
            if "perm_en" in measures:
                out["perm_en"][ch] = ant.perm_entropy(x_full, order=3,
                                                      normalize=True)
            if "spec_en" in measures:
                out["spec_en"][ch] = ant.spectral_entropy(
                    x_full, sf=sfreq, method="welch", normalize=True)
            if "higuchi" in measures:
                out["higuchi"][ch] = ant.higuchi_fd(x_full)
            if "dfa" in measures:
                out["dfa"][ch] = ant.detrended_fluctuation(x_full)
            if "lziv" in measures:
                bx = (x_lziv > np.median(x_lziv)).astype(int)
                out["lziv"][ch] = ant.lziv_complexity(bx, normalize=True)
        except Exception:
            pass
    return out


def multiscale_entropy_curve(
    data: np.ndarray, scales: Sequence[int] = tuple(range(1, 11)),
    max_samples_per_scale: int = 3000,
) -> np.ndarray:
    """
    Multiscale entropy curve per channel.
    Returns (n_channels, n_scales).

    Each scale's coarse-grained signal is capped to `max_samples_per_scale`
    samples for speed. Sample entropy stabilizes well before 5000 samples
    so this is a safe trade-off.
    """
    n_ch = data.shape[0]
    out = np.zeros((n_ch, len(scales)))
    for ch in range(n_ch):
        x = data[ch]
        for si, s in enumerate(scales):
            if s == 1:
                xs = x
            else:
                # coarse-graining: non-overlapping average
                n_seg = len(x) // s
                xs = x[:n_seg * s].reshape(n_seg, s).mean(axis=1)
            if len(xs) < 100:
                out[ch, si] = np.nan
                continue
            try:
                xs_capped = xs[:min(len(xs), max_samples_per_scale)]
                out[ch, si] = ant.sample_entropy(xs_capped, order=2)
            except Exception:
                out[ch, si] = np.nan
    return out


# =============================================================================
# 2. MICROSTATES
# =============================================================================
def _gfp(data: np.ndarray) -> np.ndarray:
    """Global field power: SD across channels at each time point. data: (n_ch, n_t)."""
    return np.std(data, axis=0, ddof=0)


def _spatial_correlation(maps_a: np.ndarray, maps_b: np.ndarray) -> np.ndarray:
    """Absolute spatial correlation between map sets. (n_a, n_ch) vs (n_b, n_ch) -> (n_a, n_b)."""
    a = maps_a - maps_a.mean(axis=1, keepdims=True)
    b = maps_b - maps_b.mean(axis=1, keepdims=True)
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return np.abs(a_n @ b_n.T)


def extract_gfp_peak_topographies(
    data: np.ndarray, min_peak_dist: int = 10,
    max_peaks: int = 5000,
) -> np.ndarray:
    """
    Extract topographies at GFP peaks. Returns (n_peaks, n_channels).
    Topographies are normalized to unit norm (polarity-invariant).
    """
    from scipy.signal import find_peaks
    gfp = _gfp(data)
    peaks, _ = find_peaks(gfp, distance=min_peak_dist)
    if len(peaks) > max_peaks:
        # Subsample by GFP magnitude (keep strongest)
        peaks = peaks[np.argsort(gfp[peaks])[-max_peaks:]]
    topos = data[:, peaks].T                              # (n_peaks, n_ch)
    # Avg-reference and normalize
    topos = topos - topos.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(topos, axis=1, keepdims=True) + 1e-12
    topos = topos / norms
    return topos


def cluster_microstate_maps(
    topographies: np.ndarray, n_maps: int = 4, n_init: int = 10,
    random_state: int = 42,
) -> np.ndarray:
    """
    Modified k-means with polarity invariance.
    We use the trick of taking |corr| as similarity by augmenting data
    with sign-flipped versions, which is equivalent for unit-normed data.

    Returns (n_maps, n_channels).
    """
    # Standard k-means in topography space; polarity invariance handled
    # at backfitting time using absolute correlation.
    km = KMeans(n_clusters=n_maps, n_init=n_init, random_state=random_state)
    km.fit(topographies)
    maps = km.cluster_centers_
    # Avg-reference, unit-norm
    maps = maps - maps.mean(axis=1, keepdims=True)
    maps /= np.linalg.norm(maps, axis=1, keepdims=True) + 1e-12
    return maps


def reorder_maps_to_canonical(
    maps: np.ndarray, info: mne.Info,
) -> Tuple[np.ndarray, List[str]]:
    """
    Try to reorder/rename the 4 microstate maps to canonical A/B/C/D
    based on rough topographic templates (left-right anterior, posterior,
    midline anterior, midline posterior).
    """
    # Heuristic: project each map onto template patterns derived from sensor pos
    pos = np.array([
        mne.channels.layout._find_topomap_coords(info, picks=[ch])[0]
        for ch in range(len(info["ch_names"]))
    ])
    x, y = pos[:, 0], pos[:, 1]
    # Templates (qualitative): A = left-occipital -> right-frontal axis
    #                         B = right-occipital -> left-frontal axis
    #                         C = anterior-posterior (midline)
    #                         D = central / radial
    templates = {
        "A": -x + y,           # left-post / right-front
        "B":  x + y,           # right-post / left-front
        "C":  y,               # anterior-posterior
        "D":  -np.abs(x) - np.abs(y),  # centrally peaked
    }
    template_arr = np.stack([templates[k] for k in ("A", "B", "C", "D")])
    template_arr -= template_arr.mean(axis=1, keepdims=True)
    template_arr /= np.linalg.norm(template_arr, axis=1, keepdims=True) + 1e-12

    sims = _spatial_correlation(maps, template_arr)       # (n_maps, 4)
    # Greedy assignment
    assigned = [None] * maps.shape[0]
    available = list(range(maps.shape[0]))
    for tmpl_idx, name in enumerate(("A", "B", "C", "D")):
        if not available:
            break
        best = available[np.argmax([sims[i, tmpl_idx] for i in available])]
        assigned[best] = name
        available.remove(best)
    # Fill any unnamed
    leftover_names = [n for n in ("A", "B", "C", "D") if n not in assigned]
    for i, a in enumerate(assigned):
        if a is None:
            assigned[i] = leftover_names.pop(0)

    # Sort maps by canonical order
    order = sorted(range(maps.shape[0]), key=lambda i: assigned[i])
    return maps[order], [assigned[i] for i in order]


def backfit_microstates(
    data: np.ndarray, maps: np.ndarray, sfreq: float,
    smooth_window: int = 3,
) -> Dict[str, np.ndarray]:
    """
    Assign each time point to the best-matching microstate (max |corr|).
    Compute coverage, occurrence, mean duration, transition probabilities.

    data: (n_ch, n_t), maps: (n_maps, n_ch)
    """
    n_ch, n_t = data.shape
    n_maps = maps.shape[0]

    # Normalize per-time-point (avg-ref + unit norm)
    d = data - data.mean(axis=0, keepdims=True)
    d_norm = d / (np.linalg.norm(d, axis=0, keepdims=True) + 1e-12)
    m_norm = maps  # already normalized
    # Absolute correlation: |maps @ d_norm|
    sim = np.abs(m_norm @ d_norm)                      # (n_maps, n_t)
    labels = sim.argmax(axis=0)                        # (n_t,)

    # Smooth labels: small temporal smoothing window (mode filter)
    if smooth_window > 1:
        from scipy.ndimage import generic_filter
        labels = generic_filter(
            labels, lambda v: np.bincount(v.astype(int)).argmax(),
            size=smooth_window
        ).astype(int)

    # Coverage (fraction of time)
    coverage = np.array([(labels == k).mean() for k in range(n_maps)])
    # Occurrence (number of contiguous segments per second)
    transitions = np.diff(labels)
    seg_starts = np.concatenate([[0], np.where(transitions != 0)[0] + 1])
    seg_labels = labels[seg_starts]
    seg_lens = np.diff(np.concatenate([seg_starts, [n_t]]))
    duration_s = n_t / sfreq
    occurrence = np.array([(seg_labels == k).sum() / duration_s
                           for k in range(n_maps)])
    mean_duration_ms = np.array([
        seg_lens[seg_labels == k].mean() / sfreq * 1000
        if (seg_labels == k).any() else np.nan
        for k in range(n_maps)
    ])
    # Transition probability matrix
    trans = np.zeros((n_maps, n_maps))
    for a, b in zip(seg_labels[:-1], seg_labels[1:]):
        trans[a, b] += 1
    row_sums = trans.sum(axis=1, keepdims=True)
    trans = trans / np.where(row_sums == 0, 1, row_sums)

    return dict(
        labels=labels, coverage=coverage, occurrence=occurrence,
        mean_duration_ms=mean_duration_ms, transition=trans,
    )


# =============================================================================
# 3. APERIODIC vs PERIODIC (FOOOF)
# =============================================================================
def fit_fooof_per_channel(
    psd: np.ndarray, freqs: np.ndarray,
    fit_range: Tuple[float, float] = (1.0, 40.0),
    peak_width_limits: Tuple[float, float] = (1.0, 8.0),
    max_n_peaks: int = 6,
) -> Dict[str, np.ndarray]:
    """
    Fit FOOOF (aperiodic + periodic) per channel.

    psd: (n_channels, n_freqs).

    Returns:
        offset:   (n_ch,)  aperiodic offset (log10)
        exponent: (n_ch,)  aperiodic exponent (1/f slope)
        r_squared: (n_ch,)
        error:    (n_ch,)
        peaks_per_band: dict band -> (n_ch, 3) [CF, PW, BW] or NaN
    """
    fg = FOOOFGroup(peak_width_limits=peak_width_limits,
                    max_n_peaks=max_n_peaks, verbose=False)
    fg.fit(freqs, psd, fit_range)
    offset = fg.get_params("aperiodic_params", "offset")
    exponent = fg.get_params("aperiodic_params", "exponent")
    r2 = fg.get_params("r_squared")
    err = fg.get_params("error")

    # Peaks per band: for each channel, find the strongest peak (if any)
    # whose CF falls in each canonical band.
    n_ch = psd.shape[0]
    peaks_per_band: Dict[str, np.ndarray] = {}
    for band, (fmin, fmax) in FREQUENCY_BANDS.items():
        peaks = np.full((n_ch, 3), np.nan)               # CF, PW, BW
        for ch in range(n_ch):
            ch_peaks = fg.get_fooof(ch).peak_params_
            if ch_peaks.size == 0:
                continue
            in_band = (ch_peaks[:, 0] >= fmin) & (ch_peaks[:, 0] <= fmax)
            if in_band.any():
                # strongest peak in band
                cands = ch_peaks[in_band]
                best = cands[np.argmax(cands[:, 1])]
                peaks[ch] = best
        peaks_per_band[band] = peaks

    return dict(offset=offset, exponent=exponent,
                r_squared=r2, error=err, peaks_per_band=peaks_per_band)


# =============================================================================
# 4. CROSS-FREQUENCY COUPLING (Modulation Index, Tort 2010)
# =============================================================================
def modulation_index(
    signal: np.ndarray, sfreq: float,
    phase_band: Tuple[float, float],
    amp_band: Tuple[float, float],
    n_bins: int = 18,
) -> float:
    """
    Tort modulation index between phase of phase_band and amplitude of amp_band.
    signal: 1D.
    """
    # Bandpass filter
    phase_sig = mne.filter.filter_data(signal, sfreq, phase_band[0],
                                       phase_band[1], verbose="ERROR")
    amp_sig = mne.filter.filter_data(signal, sfreq, amp_band[0],
                                     amp_band[1], verbose="ERROR")
    phase = np.angle(hilbert(phase_sig))
    amp = np.abs(hilbert(amp_sig))

    # Bin amplitude by phase
    bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    binned = np.zeros(n_bins)
    for k in range(n_bins):
        mask = (phase >= bins[k]) & (phase < bins[k + 1])
        if mask.any():
            binned[k] = amp[mask].mean()
    if binned.sum() == 0:
        return np.nan
    P = binned / binned.sum()
    P = np.where(P == 0, 1e-12, P)
    H = -np.sum(P * np.log(P))
    H_max = np.log(n_bins)
    return float((H_max - H) / H_max)


def cfc_per_channel(
    data: np.ndarray, sfreq: float,
    pairs: Sequence[Tuple[Tuple[float, float], Tuple[float, float], str]] = (
        ((4, 8),  (30, 50), "theta-gamma"),
        ((8, 12), (30, 50), "alpha-gamma"),
        ((4, 8),  (12, 30), "theta-beta"),
    ),
    n_bins: int = 18,
    max_samples: int = 30000,
) -> Dict[str, np.ndarray]:
    """
    Tort Modulation Index per channel for each (phase, amp) pair.

    Optimizations vs naive version:
      - Filter ALL channels at once per band (not per channel).
      - Cache band-passed signals so each unique band is filtered once,
        not once per pair.
      - Cap signal length to `max_samples` to keep filtering fast on
        very long resting-state recordings.
    """
    n_ch, n_t = data.shape
    if n_t > max_samples:
        data = data[:, :max_samples]
        n_t = max_samples

    # Identify unique bands across all pairs
    unique_bands = {}
    for phase_band, amp_band, _ in pairs:
        unique_bands[phase_band] = None
        unique_bands[amp_band]   = None

    # Filter each unique band ONCE for all channels at once
    band_filtered = {}
    for band in unique_bands:
        fmin, fmax = band
        # filter_data on (n_ch, n_t) is fully vectorized internally
        filt = mne.filter.filter_data(data, sfreq, fmin, fmax, verbose="ERROR")
        band_filtered[band] = filt

    # Compute Hilbert per band (also vectorized)
    band_phase = {b: np.angle(hilbert(s, axis=1))
                  for b, s in band_filtered.items()}
    band_amp   = {b: np.abs(hilbert(s, axis=1))
                  for b, s in band_filtered.items()}

    # Compute MI per pair per channel
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    H_max = np.log(n_bins)
    out = {label: np.zeros(n_ch) for _, _, label in pairs}

    for phase_band, amp_band, label in pairs:
        phase = band_phase[phase_band]   # (n_ch, n_t)
        amp   = band_amp[amp_band]       # (n_ch, n_t)
        # digitize: bin index per sample per channel
        bin_idx = np.digitize(phase, bin_edges) - 1   # (n_ch, n_t)
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)

        for ch in range(n_ch):
            # Mean amp per bin via bincount
            sums = np.bincount(bin_idx[ch], weights=amp[ch], minlength=n_bins)
            counts = np.bincount(bin_idx[ch], minlength=n_bins)
            with np.errstate(invalid="ignore", divide="ignore"):
                binned = np.where(counts > 0, sums / counts, 0.0)
            total = binned.sum()
            if total <= 0:
                out[label][ch] = np.nan
                continue
            P = binned / total
            P = np.where(P == 0, 1e-12, P)
            H = -np.sum(P * np.log(P))
            out[label][ch] = (H_max - H) / H_max
    return out


# =============================================================================
# 5. ASYMMETRY / LATERALIZATION
# =============================================================================
def asymmetry_indices(
    psd_per_channel: np.ndarray, freqs: np.ndarray,
    ch_names: List[str], pairs: Sequence[Tuple[str, str]] = FAA_PAIRS,
    bands: Dict[str, Tuple[float, float]] = FREQUENCY_BANDS,
) -> Dict[str, Dict[str, float]]:
    """
    Compute asymmetry index = ln(power_R) - ln(power_L) per band per pair.
    Positive values indicate greater right-hemisphere power.

    Note on FAA: classic interpretation is that **lower** frontal alpha
    asymmetry (ln(R) - ln(L) negative, i.e., relatively greater right-frontal
    alpha = relatively *less* right-frontal activity) is associated with
    withdrawal/negative affect. Increases (positive shifts) often reported
    in mindfulness/meditation studies as an approach-motivation marker.

    Returns:
        {band: {pair_label: asymmetry_value}}
    """
    out: Dict[str, Dict[str, float]] = {b: {} for b in bands}
    for band, (fmin, fmax) in bands.items():
        idx = np.logical_and(freqs >= fmin, freqs <= fmax)
        if not idx.any():
            continue
        for left, right in pairs:
            if left not in ch_names or right not in ch_names:
                continue
            pL = np.trapezoid(psd_per_channel[ch_names.index(left), idx],
                              x=freqs[idx])
            pR = np.trapezoid(psd_per_channel[ch_names.index(right), idx],
                              x=freqs[idx])
            if pL <= 0 or pR <= 0:
                out[band][f"{left}-{right}"] = np.nan
            else:
                out[band][f"{left}-{right}"] = float(np.log(pR) - np.log(pL))
    return out


# =============================================================================
# 6. INDIVIDUAL ALPHA FREQUENCY (IAF)
# =============================================================================
def individual_alpha_frequency(
    psd_per_channel: np.ndarray, freqs: np.ndarray,
    alpha_search: Tuple[float, float] = (7.0, 13.0),
    posterior_idx: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Compute individual alpha frequency:
        - PAF (peak alpha frequency): freq of max power in alpha band
        - CoG (center of gravity): power-weighted mean frequency in alpha band

    If posterior_idx given, average across those channels first
    (recommended: occipital/parietal, where alpha is strongest).

    Returns dict with 'paf', 'cog', 'alpha_power'.
    """
    fmin, fmax = alpha_search
    band_idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    if not band_idx.any():
        return dict(paf=np.nan, cog=np.nan, alpha_power=np.nan)
    band_freqs = freqs[band_idx]

    if posterior_idx is not None and len(posterior_idx) > 0:
        spec = psd_per_channel[posterior_idx][:, band_idx].mean(axis=0)
    else:
        spec = psd_per_channel[:, band_idx].mean(axis=0)

    if spec.sum() <= 0:
        return dict(paf=np.nan, cog=np.nan, alpha_power=np.nan)

    paf = float(band_freqs[np.argmax(spec)])
    cog = float(np.sum(band_freqs * spec) / np.sum(spec))
    return dict(paf=paf, cog=cog, alpha_power=float(spec.mean()))


# =============================================================================
# 7. TIME-RESOLVED SPECTRAL DYNAMICS (rolling-window PSD)
# =============================================================================
def time_resolved_band_power(
    data: np.ndarray, sfreq: float,
    bands: Dict[str, Tuple[float, float]] = FREQUENCY_BANDS,
    window_s: float = 2.0, step_s: float = 0.5,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Sliding-window band-power time series per channel.

    data: (n_ch, n_t) — typically a continuous segment per subject
                       (concatenated epochs are fine for variability).
    Returns:
        {'time': (n_windows,), <band>: (n_ch, n_windows)}
    """
    n_ch, n_t = data.shape
    win = int(window_s * sfreq)
    step = int(step_s * sfreq)
    if win > n_t:
        win = n_t
        step = n_t
    starts = np.arange(0, n_t - win + 1, step)
    times = (starts + win / 2) / sfreq

    result: Dict[str, np.ndarray] = {b: np.zeros((n_ch, len(starts)))
                                      for b in bands}
    for wi, s0 in enumerate(starts):
        seg = data[:, s0:s0 + win]
        f, p = welch(seg, fs=sfreq, axis=1, nperseg=min(win, 512))
        for b, (fmin, fmax) in bands.items():
            band_idx = np.logical_and(f >= fmin, f <= fmax)
            if band_idx.any():
                result[b][:, wi] = np.trapezoid(p[:, band_idx],
                                                x=f[band_idx], axis=1)
    return {"time": times, **result}


# =============================================================================
# 8. SOURCE LOCALIZATION (stub)
# =============================================================================
def source_localization_stub():
    """
    Source localization (eLORETA / dSPM / MNE) requires:

      1. A digitized montage matched to a head model. For your `Sujeto*`
         files you'll want to verify ch_names match a standard montage
         (e.g., 'standard_1020' or 'biosemi64'); otherwise co-register.
      2. An MRI or template ('fsaverage' from MNE).
      3. A BEM model + source space:
             mne.bem.make_watershed_bem (or use fsaverage's prebuilt BEM)
             mne.setup_source_space
             mne.make_bem_model -> mne.make_bem_solution
      4. A forward solution:
             mne.make_forward_solution
      5. Noise covariance (compute from rest baseline or empty room).
      6. Inverse operator + apply:
             mne.minimum_norm.make_inverse_operator
             mne.minimum_norm.apply_inverse_epochs (lambda2=1/SNR^2,
                                                     method='eLORETA')

    This is a separate ~300 line pipeline. Recommended template-MRI route:

        subjects_dir = mne.datasets.fetch_fsaverage()
        # ... create fwd, cov, inv, then apply per-band band-pass

    I've left this as a stub because building it without your actual
    montage details could produce silently incorrect source estimates.
    """
    raise NotImplementedError(
        "Source localization requires montage + forward model setup. "
        "See docstring for the recommended fsaverage-based pipeline."
    )


# =============================================================================
# Group-level wrapper: compute everything for one subject pair, then aggregate
# =============================================================================
@dataclass
class SubjectAdvanced:
    """Per-subject results for all advanced analyses."""
    sid: str
    info: mne.Info
    # Power-spectrum features
    psd:        np.ndarray = None     # (n_ch, n_freqs)
    freqs:      np.ndarray = None
    # Complexity
    complexity: Dict[str, np.ndarray] = field(default_factory=dict)
    mse_curve:  np.ndarray = None     # (n_ch, n_scales)
    # FOOOF
    fooof:      Dict[str, np.ndarray] = None
    # PAC
    pac:        Dict[str, np.ndarray] = None
    # Asymmetry
    asymmetry:  Dict[str, Dict[str, float]] = None
    # IAF
    iaf:        Dict[str, float] = None
    # Microstate backfit (filled later, group-shared maps)
    microstates: Dict = None


def _continuous_from_epochs(epochs: mne.BaseEpochs) -> np.ndarray:
    """Concatenate epochs along time axis -> (n_ch, n_total_t)."""
    d = epochs.get_data(copy=False)         # (n_ep, n_ch, n_t)
    return d.transpose(1, 0, 2).reshape(d.shape[1], -1)


def analyze_subject(
    epochs: mne.BaseEpochs,
    sid: str,
    fmin: float = 0.5, fmax: float = 50.0,
    do_complexity: bool = True,
    do_fooof: bool = True,
    do_pac: bool = True,
    do_mse: bool = True,
    mse_scales: Sequence[int] = tuple(range(1, 11)),
    verbose: bool = True,
) -> SubjectAdvanced:
    """Run the per-subject advanced suite (everything except microstates,
    which need group-level cluster maps)."""
    import time
    info = epochs.info
    sfreq = info["sfreq"]
    cont = _continuous_from_epochs(epochs)
    if verbose:
        print(f"     [{sid}] data shape: {cont.shape} "
              f"({cont.shape[1]/sfreq:.0f}s, {len(info['ch_names'])} ch)")

    # PSD (multitaper)
    t0 = time.time()
    psd_obj = epochs.compute_psd(method="multitaper", fmin=fmin, fmax=fmax,
                                 verbose="ERROR")
    psd, freqs = psd_obj.get_data(return_freqs=True)
    psd_mean = psd.mean(axis=0)                            # (n_ch, n_freqs)
    if verbose:
        print(f"     [{sid}] PSD ({time.time()-t0:.1f}s)")

    out = SubjectAdvanced(sid=sid, info=info, psd=psd_mean, freqs=freqs)

    if do_complexity:
        t0 = time.time()
        out.complexity = compute_complexity_per_channel(cont, sfreq)
        if verbose:
            print(f"     [{sid}] Complexity ({time.time()-t0:.1f}s)")
    if do_mse:
        t0 = time.time()
        out.mse_curve = multiscale_entropy_curve(cont, scales=mse_scales)
        if verbose:
            print(f"     [{sid}] MSE ({time.time()-t0:.1f}s)")
    if do_fooof:
        t0 = time.time()
        out.fooof = fit_fooof_per_channel(psd_mean, freqs)
        if verbose:
            print(f"     [{sid}] FOOOF ({time.time()-t0:.1f}s)")
    if do_pac:
        t0 = time.time()
        out.pac = cfc_per_channel(cont, sfreq)
        if verbose:
            print(f"     [{sid}] PAC ({time.time()-t0:.1f}s)")

    out.asymmetry = asymmetry_indices(psd_mean, freqs, info["ch_names"])
    posterior_chs = [info["ch_names"].index(c)
                     for c in ROI_DEFINITIONS["Occipital"] +
                              ROI_DEFINITIONS["Parietal"]
                     if c in info["ch_names"]]
    out.iaf = individual_alpha_frequency(
        psd_mean, freqs, posterior_idx=np.array(posterior_chs))

    return out


# =============================================================================
# Group microstates: shared maps from concatenated GFP-peak topographies
# =============================================================================
def fit_group_microstates(
    epochs_list: Sequence[mne.BaseEpochs],
    n_maps: int = 4, peaks_per_subject: int = 1000,
) -> Tuple[np.ndarray, List[str], mne.Info]:
    """
    Pool GFP-peak topographies across subjects and fit shared k-means.
    Returns (maps, labels A/B/C/D, info).
    """
    pooled = []
    info_ref = None
    for ep in epochs_list:
        cont = _continuous_from_epochs(ep)
        topos = extract_gfp_peak_topographies(cont, max_peaks=peaks_per_subject)
        pooled.append(topos)
        if info_ref is None:
            info_ref = ep.info
    pooled = np.concatenate(pooled, axis=0)
    raw_maps = cluster_microstate_maps(pooled, n_maps=n_maps)
    maps, names = reorder_maps_to_canonical(raw_maps, info_ref)
    return maps, names, info_ref


# =============================================================================
# Plotting
# =============================================================================
def _topomap(data, info, ax, **kw):
    kw.setdefault("contours", 0)
    kw.setdefault("sensors", True)
    kw.setdefault("outlines", "head")
    kw.setdefault("show", False)
    return mne.viz.plot_topomap(data, info, axes=ax, **kw)


def plot_complexity_topomaps(
    R_per_subj: List[Dict[str, np.ndarray]],
    MF_per_subj: List[Dict[str, np.ndarray]],
    info: mne.Info, save_path: Optional[str] = None,
):
    """Per-measure topomaps: R / MF / diff / Cohen's d_z."""
    measures = list(R_per_subj[0].keys())
    n_rows = 4
    n_cols = len(measures)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(2.7 * n_cols, 2.5 * n_rows),
                             constrained_layout=True)
    fig.suptitle("Complexity measures: Rest vs Mindfulness",
                 fontsize=13, fontweight="bold")
    row_titles = ["Rest", "Mindfulness", "MF − Rest", "Cohen's $d_z$"]

    for c, m in enumerate(measures):
        R = np.stack([s[m] for s in R_per_subj])           # (n_subj, n_ch)
        MF = np.stack([s[m] for s in MF_per_subj])
        R_mean, MF_mean = R.mean(0), MF.mean(0)
        diff = MF_mean - R_mean
        dz = cohens_dz(MF, R)

        vmin = min(R_mean.min(), MF_mean.min())
        vmax = max(R_mean.max(), MF_mean.max())
        vd = max(abs(diff.min()), abs(diff.max())) or 1e-12
        vdz = max(abs(np.nanmin(dz)), abs(np.nanmax(dz))) or 1e-12

        _topomap(R_mean,  info, axes[0, c], cmap="viridis", vlim=(vmin, vmax))
        _topomap(MF_mean, info, axes[1, c], cmap="viridis", vlim=(vmin, vmax))
        _topomap(diff, info, axes[2, c], cmap="RdBu_r", vlim=(-vd, vd))
        _topomap(dz,   info, axes[3, c], cmap="RdBu_r", vlim=(-vdz, vdz))
        axes[0, c].set_title(m, fontweight="bold")
        if c == 0:
            for r, t in enumerate(row_titles):
                axes[r, 0].set_ylabel(t, fontweight="bold")
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_microstate_maps(
    maps: np.ndarray, names: List[str], info: mne.Info,
    save_path: Optional[str] = None,
):
    """The 4 canonical microstate topographies."""
    n = maps.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2), constrained_layout=True)
    fig.suptitle("Group microstate maps", fontsize=13, fontweight="bold")
    for i, ax in enumerate(axes):
        vmax = np.max(np.abs(maps[i]))
        _topomap(maps[i], info, ax, cmap="RdBu_r", vlim=(-vmax, vmax))
        ax.set_title(f"Map {names[i]}", fontweight="bold")
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_microstate_metrics(
    R_metrics: List[Dict], MF_metrics: List[Dict],
    map_names: List[str], save_path: Optional[str] = None,
):
    """Bar plots of coverage / occurrence / mean duration per state."""
    keys = ["coverage", "occurrence", "mean_duration_ms"]
    titles = ["Coverage (fraction)", "Occurrence (per s)", "Mean duration (ms)"]
    n_maps = len(map_names)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    fig.suptitle("Microstate temporal dynamics", fontsize=13, fontweight="bold")

    for ax, key, t in zip(axes, keys, titles):
        R = np.stack([s[key] for s in R_metrics])         # (n_subj, n_maps)
        MF = np.stack([s[key] for s in MF_metrics])
        x = np.arange(n_maps)
        w = 0.35
        ax.bar(x - w/2, R.mean(0), w, yerr=R.std(0, ddof=1) / np.sqrt(len(R)),
               label="Rest", color="#5B8FB9", capsize=3)
        ax.bar(x + w/2, MF.mean(0), w, yerr=MF.std(0, ddof=1) / np.sqrt(len(MF)),
               label="Mindfulness", color="#B6464A", capsize=3)
        for i in range(n_maps):
            stats = paired_test_summary(R[:, i], MF[:, i])
            stars = _add_stars(stats["p"])
            if stars:
                y = max(R[:, i].mean(), MF[:, i].mean()) * 1.1
                ax.text(i, y, stars, ha="center", fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels([f"Map {n}" for n in map_names])
        ax.set_title(t, fontweight="bold")
        ax.legend()
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_fooof_summary(
    R_subjs: List[SubjectAdvanced], MF_subjs: List[SubjectAdvanced],
    info: mne.Info, save_path: Optional[str] = None,
):
    """Aperiodic exponent + offset topomaps; periodic peak prevalence per band."""
    R_exp = np.stack([s.fooof["exponent"] for s in R_subjs])
    MF_exp = np.stack([s.fooof["exponent"] for s in MF_subjs])
    R_off = np.stack([s.fooof["offset"]   for s in R_subjs])
    MF_off = np.stack([s.fooof["offset"]  for s in MF_subjs])

    fig, axes = plt.subplots(3, 4, figsize=(13, 9), constrained_layout=True)
    fig.suptitle("Aperiodic vs periodic decomposition (FOOOF)",
                 fontsize=13, fontweight="bold")

    for col, (R, MF, label) in enumerate([
        (R_off, MF_off, "Offset"),
        (R_exp, MF_exp, "Exponent (1/f slope)"),
    ]):
        Rm, MFm = R.mean(0), MF.mean(0)
        diff = MFm - Rm
        dz = cohens_dz(MF, R)
        vmin = min(Rm.min(), MFm.min()); vmax = max(Rm.max(), MFm.max())
        vd = max(abs(diff.min()), abs(diff.max())) or 1e-12

        _topomap(Rm,  info, axes[0, col], cmap="viridis", vlim=(vmin, vmax))
        _topomap(MFm, info, axes[1, col], cmap="viridis", vlim=(vmin, vmax))
        _topomap(diff, info, axes[2, col], cmap="RdBu_r", vlim=(-vd, vd))
        axes[0, col].set_title(label, fontweight="bold")

    # Right two columns: peak prevalence per band (alpha + beta as exemplars)
    for col, band in zip([2, 3], ["Alpha", "Beta"]):
        # Fraction of channels with a peak in this band
        R_prev = np.stack([
            (~np.isnan(s.fooof["peaks_per_band"][band][:, 0])).astype(float)
            for s in R_subjs])
        MF_prev = np.stack([
            (~np.isnan(s.fooof["peaks_per_band"][band][:, 0])).astype(float)
            for s in MF_subjs])
        Rm, MFm = R_prev.mean(0), MF_prev.mean(0)
        diff = MFm - Rm
        _topomap(Rm,  info, axes[0, col], cmap="magma", vlim=(0, 1))
        _topomap(MFm, info, axes[1, col], cmap="magma", vlim=(0, 1))
        vd = max(abs(diff.min()), abs(diff.max())) or 1e-12
        _topomap(diff, info, axes[2, col], cmap="RdBu_r", vlim=(-vd, vd))
        axes[0, col].set_title(f"{band} peak prevalence", fontweight="bold")

    for r, label in enumerate(["Rest", "Mindfulness", "MF − Rest"]):
        axes[r, 0].set_ylabel(label, fontweight="bold")
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_pac(
    R_subjs: List[SubjectAdvanced], MF_subjs: List[SubjectAdvanced],
    info: mne.Info, save_path: Optional[str] = None,
):
    """Cross-frequency coupling topomaps (Tort MI)."""
    pairs = list(R_subjs[0].pac.keys())
    n = len(pairs)
    fig, axes = plt.subplots(3, n, figsize=(3.2 * n, 8), constrained_layout=True)
    fig.suptitle("Cross-frequency coupling (Modulation Index)",
                 fontsize=13, fontweight="bold")
    for c, p in enumerate(pairs):
        R = np.stack([s.pac[p] for s in R_subjs])
        MF = np.stack([s.pac[p] for s in MF_subjs])
        Rm, MFm = R.mean(0), MF.mean(0)
        diff = MFm - Rm
        vmin = min(Rm.min(), MFm.min()); vmax = max(Rm.max(), MFm.max())
        vd = max(abs(diff.min()), abs(diff.max())) or 1e-12
        _topomap(Rm,  info, axes[0, c], cmap="viridis", vlim=(vmin, vmax))
        _topomap(MFm, info, axes[1, c], cmap="viridis", vlim=(vmin, vmax))
        _topomap(diff, info, axes[2, c], cmap="RdBu_r", vlim=(-vd, vd))
        axes[0, c].set_title(p, fontweight="bold")
    for r, label in enumerate(["Rest", "Mindfulness", "MF − Rest"]):
        axes[r, 0].set_ylabel(label, fontweight="bold")
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_asymmetry(
    R_subjs: List[SubjectAdvanced], MF_subjs: List[SubjectAdvanced],
    save_path: Optional[str] = None,
):
    """Bar plot of asymmetry indices per band per pair, with stats."""
    bands = list(FREQUENCY_BANDS.keys())
    pairs = list(R_subjs[0].asymmetry["Alpha"].keys())

    fig, axes = plt.subplots(1, len(bands),
                             figsize=(3 * len(bands), 4.5),
                             constrained_layout=True, sharey=True)
    fig.suptitle("Asymmetry indices (ln Right − ln Left), MF vs Rest",
                 fontsize=13, fontweight="bold")

    for ax, band in zip(axes, bands):
        R = np.array([[s.asymmetry[band].get(p, np.nan) for p in pairs]
                      for s in R_subjs])
        MF = np.array([[s.asymmetry[band].get(p, np.nan) for p in pairs]
                       for s in MF_subjs])
        x = np.arange(len(pairs))
        w = 0.35
        ax.bar(x - w/2, np.nanmean(R, 0), w,
               yerr=np.nanstd(R, 0, ddof=1) / np.sqrt(len(R)),
               label="Rest", color="#5B8FB9", capsize=3)
        ax.bar(x + w/2, np.nanmean(MF, 0), w,
               yerr=np.nanstd(MF, 0, ddof=1) / np.sqrt(len(MF)),
               label="Mindfulness", color="#B6464A", capsize=3)
        for i, p in enumerate(pairs):
            stats = paired_test_summary(R[:, i], MF[:, i])
            stars = _add_stars(stats["p"])
            if stars:
                y = max(np.nanmean(R[:, i]), np.nanmean(MF[:, i])) * 1.15
                ax.text(i, y, stars, ha="center", fontsize=11)
        ax.axhline(0, color="black", lw=0.5)
        ax.set_xticks(x); ax.set_xticklabels(pairs, rotation=30, ha="right")
        ax.set_title(band, fontweight="bold")
        if ax is axes[0]:
            ax.set_ylabel("Asymmetry index")
            ax.legend()
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_iaf(
    R_subjs: List[SubjectAdvanced], MF_subjs: List[SubjectAdvanced],
    save_path: Optional[str] = None,
):
    """IAF: paired plot of PAF and CoG."""
    paf_R  = np.array([s.iaf["paf"] for s in R_subjs])
    paf_MF = np.array([s.iaf["paf"] for s in MF_subjs])
    cog_R  = np.array([s.iaf["cog"] for s in R_subjs])
    cog_MF = np.array([s.iaf["cog"] for s in MF_subjs])

    fig, axes = plt.subplots(1, 2, figsize=(9, 5), constrained_layout=True)
    fig.suptitle("Individual alpha frequency (IAF)", fontsize=13, fontweight="bold")
    for ax, (R, MF, label) in zip(axes, [
        (paf_R, paf_MF, "Peak Alpha Frequency"),
        (cog_R, cog_MF, "Center of Gravity"),
    ]):
        for i in range(len(R)):
            color = "#B6464A" if MF[i] > R[i] else "#5B8FB9"
            ax.plot([0, 1], [R[i], MF[i]], color=color, alpha=0.5,
                    marker="o", markersize=4)
        ax.plot([0, 1], [np.nanmean(R), np.nanmean(MF)], color="black",
                marker="s", markersize=9, lw=2.5, label="Group mean")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Rest", "Mindfulness"])
        ax.set_ylabel("Frequency (Hz)")
        stats = paired_test_summary(R, MF)
        ax.set_title(f"{label}\n$d_z$={stats['dz']:+.2f}, p={stats['p']:.3f} "
                     f"{_add_stars(stats['p'])}", fontweight="bold")
        ax.legend()
        ax.set_xlim(-0.3, 1.3)
    if save_path:
        fig.savefig(save_path)
    return fig


def plot_mse_curves(
    R_subjs: List[SubjectAdvanced], MF_subjs: List[SubjectAdvanced],
    scales: Sequence[int] = tuple(range(1, 11)),
    save_path: Optional[str] = None,
):
    """Multiscale entropy: mean ± SEM curves across all channels."""
    R_curve = np.stack([s.mse_curve.mean(axis=0) for s in R_subjs])  # (n_subj, n_scales)
    MF_curve = np.stack([s.mse_curve.mean(axis=0) for s in MF_subjs])

    fig, ax = plt.subplots(figsize=(8, 5))
    s = np.array(scales)
    Rm, Re = np.nanmean(R_curve, 0), np.nanstd(R_curve, 0, ddof=1) / np.sqrt(len(R_curve))
    MFm, MFe = np.nanmean(MF_curve, 0), np.nanstd(MF_curve, 0, ddof=1) / np.sqrt(len(MF_curve))
    ax.errorbar(s, Rm, yerr=Re, label="Rest", color="#5B8FB9", lw=2,
                marker="o", capsize=3)
    ax.errorbar(s, MFm, yerr=MFe, label="Mindfulness", color="#B6464A", lw=2,
                marker="s", capsize=3)

    # Per-scale paired t-test
    from scipy.stats import ttest_rel as _tr
    for i in range(len(s)):
        valid = ~(np.isnan(R_curve[:, i]) | np.isnan(MF_curve[:, i]))
        if valid.sum() < 3:
            continue
        _, p = _tr(MF_curve[valid, i], R_curve[valid, i])
        stars = _add_stars(p)
        if stars:
            y = max(Rm[i], MFm[i]) + max(Re[i], MFe[i]) + 0.05
            ax.text(s[i], y, stars, ha="center", fontsize=11)

    ax.set_xlabel("Scale factor τ")
    ax.set_ylabel("Sample entropy")
    ax.set_title("Multiscale entropy curve (mean across channels)",
                 fontweight="bold")
    ax.legend()
    if save_path:
        fig.savefig(save_path)
    return fig


# =============================================================================
# Top-level driver
# =============================================================================
def run_advanced_analyses(
    file_pairs: Sequence[Tuple[str, str]],
    out_dir: str = "advanced_figures",
    do_complexity: bool = True,
    do_mse: bool = True,
    do_fooof: bool = True,
    do_pac: bool = True,
    do_microstates: bool = True,
    n_microstates: int = 4,
):
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 72)
    print("STEP 1: Per-subject analyses")
    print("=" * 72)
    R_subjs: List[SubjectAdvanced] = []
    MF_subjs: List[SubjectAdvanced] = []
    R_epochs_list, MF_epochs_list = [], []

    for i, (f_R, f_MF) in enumerate(file_pairs):
        sid = f"S{i + 1:02d}"
        print(f"[{sid}] Loading & analyzing ...")
        try:
            ep_R  = mne.read_epochs(f_R,  preload=True, verbose="ERROR")
            ep_MF = mne.read_epochs(f_MF, preload=True, verbose="ERROR")
        except Exception as e:
            print(f"  ! skipped: {e}")
            continue
        common = [c for c in ep_R.ch_names if c in ep_MF.ch_names]
        ep_R  = ep_R.copy().pick(common)
        ep_MF = ep_MF.copy().pick(common)

        R_subjs.append(analyze_subject(ep_R, sid + "_R",
                                       do_complexity=do_complexity,
                                       do_mse=do_mse, do_fooof=do_fooof,
                                       do_pac=do_pac))
        MF_subjs.append(analyze_subject(ep_MF, sid + "_MF",
                                        do_complexity=do_complexity,
                                        do_mse=do_mse, do_fooof=do_fooof,
                                        do_pac=do_pac))
        R_epochs_list.append(ep_R)
        MF_epochs_list.append(ep_MF)

    info = R_subjs[0].info
    print(f"  -> {len(R_subjs)} subjects analyzed")

    if do_microstates:
        print("\n" + "=" * 72)
        print("STEP 2: Group microstates")
        print("=" * 72)
        # Pool R + MF for shared maps
        maps, names, _ = fit_group_microstates(
            R_epochs_list + MF_epochs_list, n_maps=n_microstates)
        # Backfit per subject per condition
        R_ms, MF_ms = [], []
        for ep in R_epochs_list:
            cont = _continuous_from_epochs(ep)
            R_ms.append(backfit_microstates(cont, maps, ep.info["sfreq"]))
        for ep in MF_epochs_list:
            cont = _continuous_from_epochs(ep)
            MF_ms.append(backfit_microstates(cont, maps, ep.info["sfreq"]))
    else:
        maps = names = R_ms = MF_ms = None

    print("\n" + "=" * 72)
    print("STEP 3: Plots")
    print("=" * 72)

    # Complexity
    if do_complexity:
        plot_complexity_topomaps(
            [s.complexity for s in R_subjs],
            [s.complexity for s in MF_subjs], info,
            save_path=os.path.join(out_dir, "01_complexity.png"))
    if do_mse:
        plot_mse_curves(R_subjs, MF_subjs,
                        save_path=os.path.join(out_dir, "02_mse_curve.png"))
    if do_fooof:
        plot_fooof_summary(R_subjs, MF_subjs, info,
                           save_path=os.path.join(out_dir, "03_fooof.png"))
    if do_pac:
        plot_pac(R_subjs, MF_subjs, info,
                 save_path=os.path.join(out_dir, "04_pac.png"))

    plot_asymmetry(R_subjs, MF_subjs,
                   save_path=os.path.join(out_dir, "05_asymmetry.png"))
    plot_iaf(R_subjs, MF_subjs,
             save_path=os.path.join(out_dir, "06_iaf.png"))

    if do_microstates:
        plot_microstate_maps(maps, names, info,
                             save_path=os.path.join(out_dir, "07_microstate_maps.png"))
        plot_microstate_metrics(R_ms, MF_ms, names,
                                save_path=os.path.join(out_dir, "08_microstate_metrics.png"))

    plt.close("all")
    print(f"  Figures saved in: {os.path.abspath(out_dir)}")

    return dict(
        R_subjs=R_subjs, MF_subjs=MF_subjs,
        microstate_maps=maps, microstate_names=names,
        R_microstate_metrics=R_ms, MF_microstate_metrics=MF_ms,
    )


# =============================================================================
# Example
# =============================================================================
if __name__ == "__main__":
    file_pairs = [
        (f"Sujeto{i}_R_processed.fif", f"Sujeto{i}_MF_processed.fif")
        for i in list(range(1, 12)) + list(range(13, 35)) + [36, 37]
    ]
    results = run_advanced_analyses(
        file_pairs,
        out_dir="advanced_figures",
        do_complexity=True,
        do_mse=True,
        do_fooof=True,
        do_pac=True,
        do_microstates=True,
    )


C:\Users\acadena\AppData\Local\Temp\ipykernel_29376\319752119.py:73: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOFGroup


STEP 1: Per-subject analyses
[S01] Loading & analyzing ...
     [S01_R] data shape: (61, 348228) (340s, 61 ch)
     [S01_R] PSD (7.0s)
     [S01_R] Complexity (25.3s)
     [S01_R] MSE (9.3s)
     [S01_R] FOOOF (1.6s)
     [S01_R] PAC (1.4s)
     [S01_MF] data shape: (61, 742545) (725s, 61 ch)
     [S01_MF] PSD (16.0s)
     [S01_MF] Complexity (49.9s)
     [S01_MF] MSE (9.7s)
     [S01_MF] FOOOF (1.7s)
     [S01_MF] PAC (1.0s)
[S02] Loading & analyzing ...
     [S02_R] data shape: (61, 297018) (290s, 61 ch)
     [S02_R] PSD (5.9s)
     [S02_R] Complexity (22.1s)
     [S02_R] MSE (9.3s)
     [S02_R] FOOOF (1.8s)
     [S02_R] PAC (1.0s)
     [S02_MF] data shape: (61, 706698) (690s, 61 ch)
     [S02_MF] PSD (14.1s)
     [S02_MF] Complexity (46.3s)
     [S02_MF] MSE (9.5s)
     [S02_MF] FOOOF (1.1s)
     [S02_MF] PAC (0.9s)
[S03] Loading & analyzing ...
     [S03_R] data shape: (61, 312381) (305s, 61 ch)
     [S03_R] PSD (5.9s)
     [S03_R] Complexity (21.7s)
     [S03_R] MSE (8.8s)
     [S